In [49]:
# ============================================================
# 1. IMPORTS, PATHS & HYPERPARAMETERS
# ============================================================

import os
import warnings
import logging
import pandas as pd
import numpy as np
import torch
import umap
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from wordcloud import WordCloud
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

# ── LOGGER ────────────────────────────────────────────────────────────────────
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")

# ── THREADING & DEVICE ────────────────────────────────────────────────────────
os.environ.update({k: "128" for k in [
    "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"
]})
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device}")

# ── PATHS ─────────────────────────────────────────────────────────────────────
CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/BERTopic"

# ── EMBEDDING MODEL ───────────────────────────────────────────────────────────
MODEL_NAME  = "sentence-transformers/embeddinggemma-300m-medical"
embed_model = SentenceTransformer(MODEL_NAME)
# Force le chargement complet avant de continuer
_ = embed_model.encode(["test"], show_progress_bar=False)
log.info(f"Model ready: {MODEL_NAME}")

# ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
BATCH_SIZE     = 256
MAX_LENGTH     = 512

UMAP_DIM       = 10
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST  = 0.0

MCS_VALUES     = range(200, 2200, 200)
MIN_SAMPLES    = 10


2026-04-02 10:12:20,302 — INFO — Device: cuda
2026-04-02 10:12:20,306 — INFO — Use pytorch device_name: cuda:0
2026-04-02 10:12:20,306 — INFO — Load pretrained SentenceTransformer: sentence-transformers/embeddinggemma-300m-medical
2026-04-02 10:12:20,600 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 10:12:20,647 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7eedc5047b2e26bcbccbf09c435ae4a80a08d/modules.json "HTTP/1.1 200 OK"
2026-04-02 10:12:20,829 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 10:12:20,877 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7ee

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-04-02 10:12:22,614 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 10:12:22,662 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7eedc5047b2e26bcbccbf09c435ae4a80a08d/config.json "HTTP/1.1 200 OK"
2026-04-02 10:12:22,871 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 10:12:22,918 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7eedc5047b2e26bcbccbf09c435ae4a80a08d/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-02 10:12:23,123 — INFO — HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/embeddinggemma-300m-medical/tree/main/additional_chat_templates

In [50]:

# ============================================================
# 2. COLUMN DEFINITIONS & CONFIGS
# ============================================================

# ── IMAGING ───────────────────────────────────────────────────────────────────

# Boolean imaging columns {col: article + label}  — used in CONFIG_SIMPLE
IMAGING_COLS_BOOL = {
    'has_ultrasound':           "an ultrasound",
    'has_ct_scan':              "a CT scan",
    'has_xray':                 "an X-ray",
    'has_mri':                  "an MRI",
    'has_radio_interventional': "an interventional radiology procedure",
    'has_nuclear_medicine':     "a nuclear medicine imaging",
}

# Categorical imaging columns {modality: [col1, col2, ...]}  — used in CONFIG_DETAILED
IMAGING_COLS_DETAILED = {
    'ultrasound':           ['ultrasound_1', 'ultrasound_2'],
    'ct_scan':              ['ct_scan_1', 'ct_scan_2', 'ct_scan_3'],
    'xray':                 ['xray_1', 'xray_2', 'xray_3'],
    'mri':                  ['mri_1', 'mri_2'],
    'radio_interventional': ['radio_interventional_1'],
    'nuclear_medicine':     ['nuclear_medicine_1'],
}

# ── BIOLOGY ───────────────────────────────────────────────────────────────────

BIO_COLS = {
    # Venous / blood tests
    'is_hemoglobine':       "hemoglobin",
    'is_leucocytes':        "white blood cell count",
    'is_formule_leuco':     "differential leukocyte count",
    'is_urea':              "blood urea",
    'is_creatinine':        "creatinine",
    'is_sodium':            "sodium",
    'is_potassium':         "potassium",
    'is_platelets':         "platelet count",
    'is_pt':                "prothrombin time",
    'is_aptt':              "aPTT",
    'is_calcium':           "calcium",
    'is_ck':                "CK",
    'is_lactates':          "lactate",
    'is_troponine':         "troponin",
    'is_bnp':               "BNP",
    'is_ckmb':              "CK-MB",
    'is_ddimer':            "D-dimer",
    'is_crp':               "CRP",
    'is_pct':               "procalcitonin",
    'is_alat':              "ALT",
    'is_asat':              "AST",
    'is_bili_total':        "total bilirubin",
    'is_lipase':            "lipase",
    'is_alp':               "ALP",
    'is_iron':              "serum iron",
    'is_ferritin':          "ferritin",
    'is_calcium_ionized':   "ionized calcium",
    'is_aXa_aIIa':          "anti-Xa/anti-IIa activity",
    'is_fibrinogen':        "fibrinogen",
    # Special biology (has_ prefix)
    'has_blood_test':       "blood work",
    'has_culture':          "microbiological cultures",
    'has_lumbar_puncture':  "lumbar puncture",
    'has_blood_gas':        "arterial blood gas",
}

# ── PROCEDURES ────────────────────────────────────────────────────────────────

PROCEDURE_COLS = {
    'had_ekg': "an EKG",
}

# ── COLUMN GROUPS (for visualization) ────────────────────────────────────────

# Venous blood tests only (is_* prefix)
cols_venous = [
    'is_hemoglobine', 'is_leucocytes', 'is_formule_leuco', 'is_urea', 'is_creatinine',
    'is_sodium', 'is_potassium', 'is_platelets', 'is_pt', 'is_aptt',
    'is_calcium', 'is_ck', 'is_lactates', 'is_troponine', 'is_bnp',
    'is_ckmb', 'is_ddimer', 'is_crp', 'is_pct', 'is_alat',
    'is_asat', 'is_bili_total', 'is_lipase', 'is_alp', 'is_iron',
    'is_ferritin', 'is_calcium_ionized', 'is_aXa_aIIa', 'is_fibrinogen',
]

# Special biology exams (has_ prefix, not venous)
cols_special_bio = [
    'has_blood_gas',
    'has_culture',
    'has_lumbar_puncture',
]

# Categorical imaging columns (for Top3 signature in visualizations)
cols_imaging_cat = [
    'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
    'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
    'radio_interventional_1', 'nuclear_medicine_1',
]

# All binary biology columns for heatmap / radar visualizations
cols_bio_bin  = cols_venous + cols_special_bio

# Quantitative columns
cols_quanti   = ['imaging_exam_count', 'bio_exam_count']

# Alias for Top3 signature
cols_multi_cat = cols_imaging_cat

# ── SCENARIOS ─────────────────────────────────────────────────────────────────

SCENARIOS = {
    # Scenario 1: aggregated (boolean) imaging + biology flags + disposition
    "advanced_radio_bio_ekg_dispo": [
        'has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri',
        'has_radio_interventional', 'has_nuclear_medicine',
        'has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
        'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
        'imaging_exam_count', 'bio_exam_count',
    ],
    # Scenario 2: detailed venous + categorical imaging + disposition
    "advanced_detailed_radio_bio_ekg_dispo": (
        cols_venous + cols_special_bio + cols_imaging_cat + [
            'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
            'imaging_exam_count', 'bio_exam_count',
        ]
    ),
}

# ── TEXT BUILDING CONFIGS ─────────────────────────────────────────────────────

CONFIG_SIMPLE = {
    "imaging_mode":    "boolean",       # boolean has_* imaging columns
    "imaging_cols":    IMAGING_COLS_BOOL,
    "bio_cols":        BIO_COLS,
    "procedure_cols":  PROCEDURE_COLS,
    "has_disposition": True,
}

CONFIG_DETAILED = {
    "imaging_mode":    "categorical",   # string-valued categorical imaging columns
    "imaging_cols":    IMAGING_COLS_DETAILED,
    "bio_cols":        BIO_COLS,
    "procedure_cols":  PROCEDURE_COLS,
    "has_disposition": True,
}

SCENARIO_CONFIGS = {
    "advanced_radio_bio_ekg_dispo": {
        "config":  CONFIG_SIMPLE,
        "cols":    SCENARIOS["advanced_radio_bio_ekg_dispo"],
    },
    "advanced_detailed_radio_bio_ekg_dispo": {
        "config":  CONFIG_DETAILED,
        "cols":    SCENARIOS["advanced_detailed_radio_bio_ekg_dispo"],
    },
}


CLINICAL_STOP_WORDS = list(ENGLISH_STOP_WORDS) + [
    "patient", "underwent", "performed", "measured",
    "total", "had", "was", "were", "placed", "admitted",
    "transferred", "discharged", "exam", "exams",
    "the", "an", "and", "in", "biological", "imaging",
]

In [75]:
print("=== NA DISTRIBUTION IN IMAGING_COLS_DETAILED ===\n")

for modality, cols in IMAGING_COLS_DETAILED.items():
    print(f"── {modality.upper()} {'─'*40}")
    for col in cols:
        n_total   = len(df)
        n_missing = df[col].isna().sum()
        n_present = df[col].notna().sum()
        print(f"  {col:25s} | present: {n_present:5d} ({100*n_present/n_total:.1f}%) "
              f"| NA: {n_missing:5d} ({100*n_missing/n_total:.1f}%)")

        # Montre les valeurs présentes
        if n_present > 0:
            vals = df[col].dropna().value_counts().head(5)
            for val, count in vals.items():
                print(f"    → '{val}' : {count} ({100*count/n_total:.1f}%)")
    print()


=== NA DISTRIBUTION IN IMAGING_COLS_DETAILED ===

── ULTRASOUND ────────────────────────────────────────
  ultrasound_1              | present:   498 (1.7%) | NA: 29341 (98.3%)
    → 'scrotal doppler ultrasound' : 168 (0.6%)
    → 'abdomen pelvis ultrasound' : 148 (0.5%)
    → 'abdominal ultrasound' : 36 (0.1%)
    → 'venous doppler ultrasound lower limb ' : 35 (0.1%)
    → 'scrotal ultrasound' : 20 (0.1%)
  ultrasound_2              | present:    11 (0.0%) | NA: 29828 (100.0%)
    → 'scrotal ultrasound' : 3 (0.0%)
    → 'scrotal ultrasound with contrast' : 2 (0.0%)
    → 'upper limb soft tissue ultrasound' : 1 (0.0%)
    → 'pelvic ultrasound' : 1 (0.0%)
    → 'liver and biliary tract ultrasound' : 1 (0.0%)

── CT_SCAN ────────────────────────────────────────
  ct_scan_1                 | present:  7478 (25.1%) | NA: 22361 (74.9%)
    → 'ct head / brain' : 2082 (7.0%)
    → 'ct abdomen pelvis' : 1632 (5.5%)
    → 'ct brain facial bones' : 505 (1.7%)
    → 'ct angiography brain neck' : 

In [76]:
print("=== NA DISTRIBUTION IN IMAGING_COLS_BOOL ===\n")

for col, label in IMAGING_COLS_BOOL.items():
    n_total   = len(df)
    n_missing = df[col].isna().sum()
    n_zero    = (df[col] == 0).sum()
    n_one     = (df[col] == 1).sum()

    print(f"  {col:30s} ({label})")
    print(f"    1 (present) : {n_one:5d} ({100*n_one/n_total:.1f}%)")
    print(f"    0 (absent)  : {n_zero:5d} ({100*n_zero/n_total:.1f}%)")
    print(f"    NA          : {n_missing:5d} ({100*n_missing/n_total:.1f}%)")
    print()

=== NA DISTRIBUTION IN IMAGING_COLS_BOOL ===

  has_ultrasound                 (an ultrasound)
    1 (present) :   498 (1.7%)
    0 (absent)  : 29341 (98.3%)
    NA          :     0 (0.0%)

  has_ct_scan                    (a CT scan)
    1 (present) :  7478 (25.1%)
    0 (absent)  : 22361 (74.9%)
    NA          :     0 (0.0%)

  has_xray                       (an X-ray)
    1 (present) :  6736 (22.6%)
    0 (absent)  : 23103 (77.4%)
    NA          :     0 (0.0%)

  has_mri                        (an MRI)
    1 (present) :  2705 (9.1%)
    0 (absent)  : 27134 (90.9%)
    NA          :     0 (0.0%)

  has_radio_interventional       (an interventional radiology procedure)
    1 (present) :    40 (0.1%)
    0 (absent)  : 29799 (99.9%)
    NA          :     0 (0.0%)

  has_nuclear_medicine           (a nuclear medicine imaging)
    1 (present) :     2 (0.0%)
    0 (absent)  : 29837 (100.0%)
    NA          :     0 (0.0%)



In [51]:

# ============================================================
# 3. TEXT BUILDER HELPERS & FUNCTIONS
# ============================================================

def _is_present(val) -> bool:
    """Returns True if val represents a positive / present finding."""
    if val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except Exception:
        pass
    if str(val).upper() in ["NONE", "NAN", "", "0", "FALSE"]:
        return False
    return val in (1, 1.0, True) or isinstance(val, str)


def _clean_label(val):
    """Normalises a raw categorical value to a readable string, or None."""
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except Exception:
        pass
    val = str(val).strip()
    if val.upper() in ["NONE", "NAN", "", "0"]:
        return None
    return val.lower().replace("_", " ")


def _join_list(items: list) -> str:
    """['a', 'b', 'c']  ->  'a, b and c'"""
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + f" and {items[-1]}"


def _with_article(s: str) -> str:
    """Prepends 'an' or 'a' based on first letter."""
    return f"an {s}" if s[0] in "aeiouAEIOU" else f"a {s}"


def _imaging_sentence_boolean(row, imaging_cols: dict, active_cols: list = None) -> str | None:
    done = [
        label for col, label in imaging_cols.items()
        if _is_present(row.get(col))
        and (active_cols is None or col in active_cols)  # <-- filtre
    ]
    if not done:
        return None
    return f"The patient underwent {_join_list(done)}."


def _imaging_sentence_categorical(row, imaging_cols: dict, active_cols: list = None) -> str | None:
    exams = []
    for modality, cols in imaging_cols.items():
        for col in cols:
            if active_cols is not None and col not in active_cols:  # <-- filtre
                continue
            val = _clean_label(row.get(col))
            if val is not None:
                exams.append(_with_article(val))
    if not exams:
        return None
    return f"The patient underwent {_join_list(exams)}."


def build_clinical_text(row, config: dict, active_cols: list = None) -> str:
    sentences = []

    # ── 1. IMAGING ───────────────────────────────────────────────
    if config["imaging_mode"] == "categorical":
        imaging_sentence = _imaging_sentence_categorical(row, config["imaging_cols"], active_cols)
    else:
        imaging_sentence = _imaging_sentence_boolean(row, config["imaging_cols"], active_cols)

    imaging_count = int(row.get('imaging_exam_count') or 0)
    if imaging_sentence:
        sentences.append(imaging_sentence)
    elif imaging_count > 0:
        sentences.append(f"The patient underwent {imaging_count} imaging exam(s).")

    # ── 2. BIOLOGICAL EXAMS ──────────────────────────────────────
    # Filtre sur active_cols si fourni
    bio_cols_active = {
        col: label for col, label in config["bio_cols"].items()
        if active_cols is None or col in active_cols
    }
    bio_done  = [label for col, label in bio_cols_active.items() if _is_present(row.get(col))]
    bio_count = int(row.get('bio_exam_count') or 0)

    if bio_done:
        verb = "were" if len(bio_done) > 1 else "was"
        sentences.append(f"{_join_list(bio_done).capitalize()} {verb} measured.")
    elif bio_count > 0:
        sentences.append(f"{bio_count} biological exam(s) were performed.")

    # ── 3. EXAM COUNT SUMMARY ────────────────────────────────────
    count_parts = []
    if imaging_count > 0:
        count_parts.append(f"{imaging_count} imaging exam(s)")
    if bio_count > 0:
        count_parts.append(f"{bio_count} biological exam(s)")
    if count_parts:
        sentences.append(f"In total, the patient had {_join_list(count_parts)}.")

    # ── 4. PROCEDURES ────────────────────────────────────────────
    proc_cols_active = {
        col: label for col, label in config["procedure_cols"].items()
        if active_cols is None or col in active_cols
    }
    procedures = [label for col, label in proc_cols_active.items() if _is_present(row.get(col))]
    if procedures:
        sentences.append(f"The patient had {_join_list(procedures)} performed.")

    # ── 5. DISPOSITION ───────────────────────────────────────────
    if config.get("has_disposition"):
        observation  = _is_present(row.get('observation_unit'))
        hospitalized = _is_present(row.get('hospitalization'))
        transferred  = _is_present(row.get('inter_facility_transfer'))

        hosp_val = row.get('hospitalization')
        try:
            discharged = (not pd.isna(hosp_val)) and (hosp_val == 0)
        except Exception:
            discharged = False

        if observation and hospitalized:
            sentences.append("The patient was placed in the observation unit and then admitted to the hospital.")
        elif observation and transferred:
            sentences.append("The patient was placed in the observation unit and then transferred to another facility.")
        elif observation:
            sentences.append("The patient was placed in the observation unit.")
        elif hospitalized:
            sentences.append("The patient was admitted to the hospital.")
        elif transferred:
            sentences.append("The patient was transferred to another facility.")
        elif discharged:
            sentences.append("The patient was discharged.")

    return " ".join(sentences) if sentences else "No significant resource utilization was documented."

In [52]:

# ============================================================
# 4. CLUSTERING METRICS FUNCTIONS
# ============================================================

def calculate_metrics(umap_embeddings, labels, hdbscan_model):
    """
    Computes clustering quality metrics:
      - Silhouette score  : separation between clusters (higher = better)
      - Stability         : mean HDBSCAN cluster persistence (higher = denser)
      - Outlier rate      : fraction of noise points (-1)
      - Combined score    : weighted combination of the three metrics

    Returns (silhouette, stability, outlier_rate, combined_score).
    Returns zeros if clustering is degenerate (< 2 clusters).
    """
    mask       = labels != -1
    n_valid    = np.sum(mask)
    n_clusters = len(np.unique(labels[mask]))

    if n_valid < 2 or n_clusters < 2:
        log.warning("Degenerate clustering — returning zero scores")
        return 0.0, 0.0, 1.0, 0.0

    silhouette   = silhouette_score(umap_embeddings[mask], labels[mask], sample_size=3000)
    outlier_rate = np.sum(labels == -1) / len(labels)
    stability    = np.mean(hdbscan_model.cluster_persistence_)
    combined     = (silhouette * 0.4) + (stability * 0.4) + ((1 - outlier_rate) * 0.2)

    return silhouette, stability, outlier_rate, combined

In [53]:

# ============================================================
#  5. VISUALIZATION FUNCTIONS
# ============================================================

def plot_wordclouds(df_valid, sc_name, out_dir):
    """Generates one wordcloud per cluster from clinical text."""
    clusters = sorted(df_valid["cluster"].unique())
    n        = len(clusters)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, cl in zip(axes, clusters):
        text = " ".join(df_valid[df_valid["cluster"] == cl]["clinical_text"].tolist())
        wc   = WordCloud(width=600, height=400, background_color="white").generate(text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Cluster {cl} (n={len(df_valid[df_valid['cluster']==cl])})")
    plt.suptitle(f"Wordclouds — {sc_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"wordclouds_{sc_name}.png"), dpi=120)
    plt.close()
    log.info(f"Wordclouds saved — {sc_name}")


def plot_heatmap(df_valid, c_bio, sc_name, out_dir):
    """Heatmap of binary bio feature frequency per cluster."""
    heat = df_valid.groupby("cluster")[c_bio].mean() * 100
    plt.figure(figsize=(max(12, len(c_bio) * 0.5), max(6, len(heat) * 0.6)))
    sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.3, cbar_kws={"label": "%"})
    plt.title(f"Bio Feature Frequency per Cluster — {sc_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"heatmap_{sc_name}.png"), dpi=120)
    plt.close()
    log.info(f"Heatmap saved — {sc_name}")


def plot_radar(df_valid, c_bio, sc_name, out_dir):
    """Radar chart of binary bio profile per cluster."""
    means    = df_valid.groupby("cluster")[c_bio].mean() * 100
    cats     = c_bio
    n_cats   = len(cats)
    angles   = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
    angles  += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    for cl, row in means.iterrows():
        values  = row.tolist() + row.tolist()[:1]
        ax.plot(angles, values, label=f"Cluster {cl}")
        ax.fill(angles, values, alpha=0.1)
    ax.set_thetagrids(np.degrees(angles[:-1]), cats, fontsize=7)
    ax.set_title(f"Bio Radar — {sc_name}", fontsize=13, fontweight="bold")
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"radar_{sc_name}.png"), dpi=120)
    plt.close()
    log.info(f"Radar chart saved — {sc_name}")


def plot_umap2d(umap_embeddings, best_labels, best_config, sc_name, out_dir):
    """
    Projects UMAP embeddings into 2D (global + local views) and plots clusters.
    Uses euclidean metric because embeddings are already UMAP-reduced.
    """
    log.info("UMAP 2D — global view (n_neighbors=30)...")
    umap_global = umap.UMAP(
        n_components=2, n_neighbors=30,
        min_dist=0.1, metric="euclidean", random_state=42
    ).fit_transform(umap_embeddings)

    log.info("UMAP 2D — local view (n_neighbors=10)...")
    umap_local = umap.UMAP(
        n_components=2, n_neighbors=10,
        min_dist=0.01, metric="euclidean", random_state=42
    ).fit_transform(umap_embeddings)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    title = (
        f"BEST MODEL : {sc_name.upper()}\n"
        f"Clusters: {int(best_config['n_clusters'])} | "
        f"MCS: {int(best_config['mcs'])} | "
        f"Silhouette: {best_config['silhouette']:.3f} | "
        f"Stability: {best_config['stability']:.3f} | "
        f"Outliers: {best_config['outlier_rate']*100:.1f}%"
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for ax, coord, name in [
        (ax1, umap_global, "UMAP Global (n_neighbors=30)"),
        (ax2, umap_local,  "UMAP Local  (n_neighbors=10)"),
    ]:
        ax.scatter(
            coord[best_labels == -1, 0], coord[best_labels == -1, 1],
            s=1, color="lightgrey", alpha=0.2, label="Noise"
        )
        for cluster_id in sorted(set(best_labels) - {-1}):
            ax.scatter(
                coord[best_labels == cluster_id, 0],
                coord[best_labels == cluster_id, 1],
                s=4, alpha=0.6, label=f"C{cluster_id}"
            )
        ax.set_title(name)
        ax.legend(markerscale=3, fontsize=8, loc="best")

    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    plt.savefig(os.path.join(out_dir, f"{sc_name}_umap_final.png"), dpi=150)
    plt.close()
    log.info(f"UMAP 2D plot saved — {sc_name}")


def build_cluster_signatures(df_valid, c_bio, c_multi, c_quanti, df_total_len):
    """
    Builds a cluster signature DataFrame with:
      - Binary bio feature frequency (%)
      - Top 3 most frequent categorical imaging values per cluster
      - Quantitative features mean ± std
      - Patient count and weight
    """
    parts = []

    if c_bio:
        parts.append(df_valid.groupby("cluster")[c_bio].mean() * 100)

    for col in c_multi:
        ct = pd.crosstab(df_valid["cluster"], df_valid[col], normalize="index") * 100
        def get_top3(row):
            top = row.sort_values(ascending=False).head(3)
            return " | ".join([f"{n} ({v:.1f}%)" for n, v in top.items() if v > 0])
        parts.append(pd.DataFrame(ct.apply(get_top3, axis=1), columns=[f"top3_{col}"]))

    if c_quanti:
        quanti_df = pd.DataFrame(index=df_valid.groupby("cluster")[c_quanti].mean().index)
        for col in c_quanti:
            m = df_valid.groupby("cluster")[col].mean().round(1)
            s = df_valid.groupby("cluster")[col].std().round(1)
            quanti_df[f"{col}_mean±std"] = m.astype(str) + " ± " + s.astype(str)
        parts.append(quanti_df)

    signatures               = pd.concat(parts, axis=1) if parts else pd.DataFrame()
    counts                   = df_valid["cluster"].value_counts()
    signatures["n_patients"] = counts
    signatures["weight_pct"] = (counts / df_total_len * 100).round(1)
    if c_bio:
        signatures[c_bio]    = signatures[c_bio].round(1)

    return signatures.sort_values("n_patients", ascending=False)

In [54]:
# ============================================================
#  DATA LOADING
# ============================================================

df = pd.read_csv(CSV_PATH)
log.info(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# df['clinical_text'] = df.apply(
#     lambda row: build_clinical_text(row, ACTIVE_CONFIG), axis=1
# )
#
# log.info(f"Clinical text built — {df['clinical_text'].notna().sum()} rows")
# log.info(f"Example:\n{df['clinical_text'].iloc[0]}")
#
# n_empty = (df['clinical_text'] == "No significant resource utilization was documented.").sum()
# log.info(f"Rows with no utilization documented: {n_empty} ({100 * n_empty / len(df):.1f}%)")

2026-04-02 10:12:29,337 — INFO — Dataset loaded: 29839 rows, 179 columns


In [55]:
# ============================================================
# RUN SCENARIOS
# ============================================================



for sc_name, sc_def in SCENARIO_CONFIGS.items():
    config      = sc_def["config"]
    active_cols = sc_def["cols"]


    log.info(f"\n{'='*60}\nSCENARIO: {sc_name}\n{'='*60}")
    sc_dir = os.path.join(OUTPUT_DIR, sc_name)
    os.makedirs(sc_dir, exist_ok=True)

    # ── Cell 6 : build text ───────────────────────────────────
    df['clinical_text'] = df.apply(
        lambda row: build_clinical_text(row, config), axis=1
    )

    # ── Cell 7 : embeddings + UMAP + sweep ───────────────────
    texts      = df['clinical_text'].tolist()
    embeddings = embed_model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True)
    np.save(os.path.join(sc_dir, "embeddings.npy"), embeddings)

    umap_model      = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIM,
                                min_dist=UMAP_MIN_DIST, metric="cosine", random_state=42)
    umap_embeddings = umap_model.fit_transform(embeddings)
    np.save(os.path.join(sc_dir, "umap_embeddings.npy"), umap_embeddings)

    sweep_results = []
    for mcs in tqdm(MCS_VALUES, desc=f"MCS sweep — {sc_name}"):
        hdbscan_sweep = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=MIN_SAMPLES,
                                         metric="euclidean", gen_min_span_tree=True)
        labels     = hdbscan_sweep.fit_predict(umap_embeddings)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        silhouette, stability, outlier_rate, combined = calculate_metrics(
            umap_embeddings, labels, hdbscan_sweep
        )
        sweep_results.append({"mcs": mcs, "n_clusters": n_clusters,
                               "silhouette": silhouette, "stability": stability,
                               "outlier_rate": outlier_rate, "combined_score": combined})

    df_sweep  = pd.DataFrame(sweep_results)
    df_sweep.to_csv(os.path.join(sc_dir, "sweep_report.csv"), index=False)
    best_row  = df_sweep.loc[df_sweep["combined_score"].idxmax()]
    BEST_MCS  = int(best_row["mcs"])
    log.info(f"Best MCS for {sc_name}: {BEST_MCS}")






    # ── Cell 8 : final BERTopic ───────────────────────────────
    topic_model = BERTopic(
        umap_model=umap.UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIM,
                             min_dist=UMAP_MIN_DIST, metric="cosine", random_state=42),
        hdbscan_model=hdbscan.HDBSCAN(min_cluster_size=BEST_MCS, min_samples=MIN_SAMPLES,
                                       metric="euclidean", gen_min_span_tree=True,
                                       prediction_data=True),
        vectorizer_model=CountVectorizer(ngram_range=(1, 2),
                                         stop_words=CLINICAL_STOP_WORDS, min_df=5),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        embedding_model=embed_model,
        nr_topics="none",
        verbose=True,
    )
    topics, probs      = topic_model.fit_transform(texts)
    df['topic']        = topics
    df['topic_prob']   = [p.max() if hasattr(p, 'max') else p for p in probs]

    topic_model.save(os.path.join(sc_dir, "bertopic_model"))
    df.to_csv(os.path.join(sc_dir, "df_with_topics.csv"), index=False)
    topic_model.get_topic_info().to_csv(os.path.join(sc_dir, "topic_info.csv"), index=False)

    # ── Cell 9 : visualizations ───────────────────────────────
    out_dir  = os.path.join(sc_dir, "visualizations")
    os.makedirs(out_dir, exist_ok=True)
    df_valid = df[df['topic'] != -1].copy()
    df_valid['cluster'] = df_valid['topic']

    c_bio    = [c for c in cols_bio_bin   if c in df_valid.columns]
    c_multi  = [c for c in cols_multi_cat if c in df_valid.columns]
    c_quanti = [c for c in cols_quanti    if c in df_valid.columns]

    plot_wordclouds(df_valid, sc_name, out_dir)
    if c_bio:
        plot_heatmap(df_valid, c_bio, sc_name, out_dir)
        plot_radar(df_valid, c_bio, sc_name, out_dir)
    plot_umap2d(umap_embeddings, df['topic'].values, best_row.to_dict(), sc_name, out_dir)

    signatures = build_cluster_signatures(df_valid, c_bio, c_multi, c_quanti, len(df))
    signatures.to_csv(os.path.join(out_dir, "cluster_signatures.csv"))

    log.info(f"Scenario {sc_name} complete — results saved to {sc_dir}")

log.info("All scenarios complete!")

2026-04-02 10:12:29,352 — INFO — 
SCENARIO: advanced_radio_bio_ekg_dispo


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

MCS sweep — advanced_radio_bio_ekg_dispo: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:13<00:00,  1.32s/it]
2026-04-02 10:14:10,079 — INFO — Best MCS for advanced_radio_bio_ekg_dispo: 200
2026-04-02 10:14:10,093 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/933 [00:00<?, ?it/s]

2026-04-02 10:15:05,814 - BERTopic - Embedding - Completed ✓
2026-04-02 10:15:05,816 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-02 10:16:00,395 - BERTopic - Dimensionality - Completed ✓
2026-04-02 10:16:00,399 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-02 10:16:01,781 - BERTopic - Cluster - Completed ✓
2026-04-02 10:16:01,782 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-02 10:16:02,344 - BERTopic - Representation - Completed ✓
2026-04-02 10:16:02,344 - BERTopic - Topic reduction - Reducing number of topics
2026-04-02 10:16:02,361 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-02 10:16:02,915 - BERTopic - Representation - Completed ✓
2026-04-02 10:16:02,918 - BERTopic - Topic reduction - Reduced number of topics from 48 to 11
2026-04-02 10:16:03,099 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

MCS sweep — advanced_detailed_radio_bio_ekg_dispo: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:12<00:00,  1.20s/it]
2026-04-02 10:19:20,181 — INFO — Best MCS for advanced_detailed_radio_bio_ekg_dispo: 200
2026-04-02 10:19:20,191 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/933 [00:00<?, ?it/s]

2026-04-02 10:20:16,970 - BERTopic - Embedding - Completed ✓
2026-04-02 10:20:16,972 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-02 10:21:07,602 - BERTopic - Dimensionality - Completed ✓
2026-04-02 10:21:07,606 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-02 10:21:08,958 - BERTopic - Cluster - Completed ✓
2026-04-02 10:21:08,959 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-02 10:21:09,531 - BERTopic - Representation - Completed ✓
2026-04-02 10:21:09,532 - BERTopic - Topic reduction - Reducing number of topics
2026-04-02 10:21:09,548 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-02 10:21:10,181 - BERTopic - Representation - Completed ✓
2026-04-02 10:21:10,184 - BERTopic - Topic reduction - Reduced number of topics from 41 to 5
2026-04-02 10:21:10,303 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make 

In [56]:


# # ============================================================
# #  EXPLORATION: EMBEDDINGS + UMAP + HDBSCAN SWEEP
# # ============================================================
#
# texts = df['clinical_text'].tolist()
# sc_dir = os.path.join(OUTPUT_DIR, "exploration")
# os.makedirs(sc_dir, exist_ok=True)
#
# # ── STEP A : Compute embeddings (once) ───────────────────────
# log.info("Computing embeddings...")
# embeddings = embed_model.encode(
#     texts,
#     batch_size=BATCH_SIZE,
#     show_progress_bar=True,
#     device=device,
# )
# np.save(os.path.join(sc_dir, "embeddings.npy"), embeddings)
# log.info(f"Embeddings shape: {embeddings.shape}")
#
# # ── STEP B : UMAP reduction ───────────────────────────────────
# log.info(f"UMAP reduction -> {UMAP_DIM}D (cosine)...")
# umap_model = umap.UMAP(
#     n_neighbors=UMAP_NEIGHBORS,
#     n_components=UMAP_DIM,
#     min_dist=UMAP_MIN_DIST,
#     metric="cosine",
#     random_state=42,
# )
# umap_embeddings = umap_model.fit_transform(embeddings)
# np.save(os.path.join(sc_dir, "umap_embeddings.npy"), umap_embeddings)
# log.info(f"UMAP embeddings shape: {umap_embeddings.shape}")

# # ── STEP C : HDBSCAN sweep ────────────────────────────────────
# log.info("Sweeping HDBSCAN min_cluster_size...")
# sweep_results = []
#
# for mcs in tqdm(MCS_VALUES, desc="MCS sweep"):
#     hdbscan_sweep = hdbscan.HDBSCAN(
#         min_cluster_size=mcs,
#         min_samples=MIN_SAMPLES,
#         metric="euclidean",
#         gen_min_span_tree=True,
#     )
#     labels    = hdbscan_sweep.fit_predict(umap_embeddings)
#     n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
#
#     silhouette, stability, outlier_rate, combined = calculate_metrics(
#         umap_embeddings, labels, hdbscan_sweep
#     )
#
#     sweep_results.append({
#         "mcs":           mcs,
#         "n_clusters":    n_clusters,
#         "silhouette":    silhouette,
#         "stability":     stability,
#         "outlier_rate":  outlier_rate,
#         "combined_score": combined,
#     })
#     log.info(
#         f"MCS={mcs:4d} | clusters={n_clusters:3d} | "
#         f"noise={outlier_rate*100:.1f}% | sil={silhouette:.3f} | combined={combined:.3f}"
#     )
#
# df_sweep = pd.DataFrame(sweep_results)
# df_sweep.to_csv(os.path.join(sc_dir, "sweep_report.csv"), index=False)
#
# # ── STEP D : Select best MCS ──────────────────────────────────
# best_row  = df_sweep.loc[df_sweep["combined_score"].idxmax()]
# BEST_MCS  = int(best_row["mcs"])
# log.info(
#     f"Best MCS: {BEST_MCS} | clusters={int(best_row['n_clusters'])} | "
#     f"combined={best_row['combined_score']:.3f}"
# )

# # ── STEP E : Performance curves ───────────────────────────────
# plt.figure(figsize=(12, 7))
# plt.plot(df_sweep["mcs"], df_sweep["combined_score"], "k-o", label="Combined Score", linewidth=3)
# plt.plot(df_sweep["mcs"], df_sweep["silhouette"],     "r--", label="Silhouette")
# plt.plot(df_sweep["mcs"], df_sweep["stability"],      "b--", label="Stability")
# plt.plot(df_sweep["mcs"], 1 - df_sweep["outlier_rate"], "g--", label="1 - Outlier Rate")
# plt.axvline(x=BEST_MCS, color="orange", linestyle=":", label=f"Best MCS ({BEST_MCS})")
# plt.title("HDBSCAN Sweep — Clustering Optimization")
# plt.xlabel("Min Cluster Size (MCS)")
# plt.ylabel("Score")
# plt.legend()
# plt.grid(True)
# plt.savefig(os.path.join(sc_dir, "sweep_performance_curves.png"))
# plt.show()
# log.info("Exploration complete — check sweep_report.csv to confirm BEST_MCS")

In [57]:
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS # a enlever
# # ============================================================
# # FINAL RUN: BERTOPIC WITH BEST MCS
# # ============================================================
# # Run this cell once BEST_MCS is confirmed from the sweep above.
# # BERTopic recomputes embeddings internally (option B) — clean single call.
#
# log.info(f"Fitting BERTopic with MCS={BEST_MCS}...")
# os.makedirs(OUTPUT_DIR, exist_ok=True)
#
# hdbscan_final = hdbscan.HDBSCAN(
#     min_cluster_size=BEST_MCS,
#     min_samples=MIN_SAMPLES,
#     metric="euclidean",
#     gen_min_span_tree=True,
#     prediction_data=True,
# )
#
# umap_final = umap.UMAP(
#     n_neighbors=UMAP_NEIGHBORS,
#     n_components=UMAP_DIM,
#     min_dist=UMAP_MIN_DIST,
#     metric="cosine",
#     random_state=42,
# )
#
# CLINICAL_STOP_WORDS = list(ENGLISH_STOP_WORDS) + [
#     "patient", "underwent", "performed", "measured",
#     "total", "had", "was", "were", "placed", "admitted",
#     "transferred", "discharged", "exam", "exams",
#     "the", "an", "and", "in", "biological", "imaging",
# ]
#
# vectorizer_final = CountVectorizer(
#     ngram_range=(1, 2),
#     stop_words=CLINICAL_STOP_WORDS,
#     min_df=5,
# )
#
# topic_model = BERTopic(
#     umap_model=umap_final,
#     hdbscan_model=hdbscan_final,
#     vectorizer_model=vectorizer_final,
#     ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
#     embedding_model=embed_model,
#     nr_topics="auto",
#     verbose=True,
# )
#
# topics, probs  = topic_model.fit_transform(texts)
# df['topic']    = topics
# df['topic_prob'] = [p.max() if hasattr(p, 'max') else p for p in probs]
#
# n_topics = len(set(topics)) - (1 if -1 in topics else 0)
# n_noise  = (pd.Series(topics) == -1).sum()
# log.info(f"Topics found : {n_topics}")
# log.info(f"Noise (-1)   : {n_noise} rows ({100 * n_noise / len(df):.1f}%)")
#
# # ── Save ─────────────────────────────────────────────────────
# topic_model.save(os.path.join(OUTPUT_DIR, "bertopic_model"))
# df.to_csv(os.path.join(OUTPUT_DIR, "df_with_topics.csv"), index=False)
# topic_model.get_topic_info().to_csv(os.path.join(OUTPUT_DIR, "topic_info.csv"), index=False)
# topic_model.visualize_barchart(top_n_topics=20).write_html(
#     os.path.join(OUTPUT_DIR, "viz_topic_barchart.html")
# )
# log.info(f"Model and results saved to {OUTPUT_DIR}")
#
#


In [58]:
# # ============================================================
# #  VISUALIZATIONS
# # ============================================================
#
# def run_visualizations():
#     out_dir = os.path.join(OUTPUT_DIR, "visualizations")
#     os.makedirs(out_dir, exist_ok=True)
#
#     df_viz           = pd.read_csv(os.path.join(OUTPUT_DIR, "df_with_topics.csv"))
#     df_viz["cluster"] = df_viz["topic"]
#     df_valid          = df_viz[df_viz["cluster"] != -1].copy()
#     log.info(f"Visualization — {len(df_valid)} valid patients (noise excluded)")
#
#     # Available column subsets for this dataset
#     c_bio    = [c for c in cols_bio_bin    if c in df_valid.columns]
#     c_multi  = [c for c in cols_multi_cat  if c in df_valid.columns]
#     c_quanti = [c for c in cols_quanti     if c in df_valid.columns]
#
#     # ── Wordclouds ────────────────────────────────────────────
#     plot_wordclouds(df_valid, "final", out_dir)
#
#     # ── Heatmap ───────────────────────────────────────────────
#     if c_bio:
#         plot_heatmap(df_valid, c_bio, "final", out_dir)
#
#     # ── Radar chart ───────────────────────────────────────────
#     if c_bio:
#         plot_radar(df_valid, c_bio, "final", out_dir)
#
#     # ── UMAP 2D ───────────────────────────────────────────────
#     umap_emb  = np.load(os.path.join(sc_dir, "umap_embeddings.npy"))
#     best_labels = df_viz["cluster"].values
#     best_config = {
#         "n_clusters":    n_topics,
#         "mcs":           BEST_MCS,
#         "silhouette":    best_row["silhouette"],
#         "stability":     best_row["stability"],
#         "outlier_rate":  best_row["outlier_rate"],
#     }
#     plot_umap2d(umap_emb, best_labels, best_config, "final", out_dir)
#
#     # ── Cluster signatures ────────────────────────────────────
#     signatures = build_cluster_signatures(df_valid, c_bio, c_multi, c_quanti, len(df_viz))
#     signatures.to_csv(os.path.join(out_dir, "cluster_signatures.csv"))
#     log.info(f"Signatures saved — {len(signatures)} clusters")
#
#     log.info("Visualizations complete!")
#
#
# run_visualizations()

In [59]:
# # ============================================================
# # CELL 1 — IMPORTS, PATHS & HYPERPARAMETERS
# # ============================================================
#
# import os
# import warnings
# import logging
# import pandas as pd
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# import seaborn as sns
# from tqdm import tqdm
# from wordcloud import WordCloud
# from bertopic import BERTopic
# from bertopic.vectorizers import ClassTfidfTransformer
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.metrics import silhouette_score
# from sentence_transformers import SentenceTransformer
#
# warnings.filterwarnings("ignore")
#
# # ── LOGGER ────────────────────────────────────────────────────────────────────
# log = logging.getLogger(__name__)
# logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
#
# # ── THREADING & DEVICE ────────────────────────────────────────────────────────
# os.environ.update({k: "128" for k in [
#     "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
#     "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"
# ]})
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# log.info(f"Device: {device}")
#
# # ── PATHS ─────────────────────────────────────────────────────────────────────
# CSV_PATH   = "df_final_binaire_imputed.csv"
# OUTPUT_DIR = "Results/BERTopic"
#
# # ── EMBEDDING MODEL ───────────────────────────────────────────────────────────
# MODEL_NAME  = "sentence-transformers/embeddinggemma-300m-medical"
# embed_model = SentenceTransformer(MODEL_NAME)
#
# # ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
# BATCH_SIZE  = 128
# MAX_LENGTH  = 512
#
# UMAP_DIM       = 10
# UMAP_NEIGHBORS = 15
# UMAP_MIN_DIST  = 0.0
#
# MCS_VALUES  = range(200, 2200, 200)
# MIN_SAMPLES = 10
#
#
# # ============================================================
# # CELL 2 — COLUMN DEFINITIONS
# # ============================================================
#
# # ── IMAGING ───────────────────────────────────────────────────────────────────
#
# # Boolean imaging columns {col: article + label}
# IMAGING_COLS_BOOL = {
#     'has_ultrasound':           "an ultrasound",
#     'has_ct_scan':              "a CT scan",
#     'has_xray':                 "an X-ray",
#     'has_mri':                  "an MRI",
#     'has_radio_interventional': "an interventional radiology procedure",
#     'has_nuclear_medicine':     "a nuclear medicine imaging",
# }
#
# # Categorical imaging columns {modality: [col1, col2, ...]}
# IMAGING_COLS_DETAILED = {
#     'ultrasound':           ['ultrasound_1', 'ultrasound_2'],
#     'ct_scan':              ['ct_scan_1', 'ct_scan_2', 'ct_scan_3'],
#     'xray':                 ['xray_1', 'xray_2', 'xray_3'],
#     'mri':                  ['mri_1', 'mri_2'],
#     'radio_interventional': ['radio_interventional_1'],
#     'nuclear_medicine':     ['nuclear_medicine_1'],
# }
#
# # ── BIOLOGY ───────────────────────────────────────────────────────────────────
#
# BIO_COLS = {
#     'is_hemoglobine':       "hemoglobin",
#     'is_leucocytes':        "white blood cell count",
#     'is_formule_leuco':     "differential leukocyte count",
#     'is_urea':              "blood urea",
#     'is_creatinine':        "creatinine",
#     'is_sodium':            "sodium",
#     'is_potassium':         "potassium",
#     'is_platelets':         "platelet count",
#     'is_pt':                "prothrombin time",
#     'is_aptt':              "aPTT",
#     'is_calcium':           "calcium",
#     'is_ck':                "CK",
#     'is_lactates':          "lactate",
#     'is_troponine':         "troponin",
#     'is_bnp':               "BNP",
#     'is_ckmb':              "CK-MB",
#     'is_ddimer':            "D-dimer",
#     'is_crp':               "CRP",
#     'is_pct':               "procalcitonin",
#     'is_alat':              "ALT",
#     'is_asat':              "AST",
#     'is_bili_total':        "total bilirubin",
#     'is_lipase':            "lipase",
#     'is_alp':               "ALP",
#     'is_iron':              "serum iron",
#     'is_ferritin':          "ferritin",
#     'is_calcium_ionized':   "ionized calcium",
#     'is_aXa_aIIa':          "anti-Xa/anti-IIa activity",
#     'is_fibrinogen':        "fibrinogen",
#     'has_blood_test':       "blood work",
#     'has_culture':          "microbiological cultures",
#     'has_lumbar_puncture':  "lumbar puncture",
#     'has_blood_gas':        "arterial blood gas",
# }
#
# # ── PROCEDURES ────────────────────────────────────────────────────────────────
#
# PROCEDURE_COLS = {
#     'had_ekg': "an EKG",
# }
#
# # ── COLUMN GROUPS (for visualization / scenarios) ─────────────────────────────
#
# cols_venous = [
#     'is_hemoglobine', 'is_leucocytes', 'is_formule_leuco', 'is_urea', 'is_creatinine',
#     'is_sodium', 'is_potassium', 'is_platelets', 'is_pt', 'is_aptt',
#     'is_calcium', 'is_ck', 'is_lactates', 'is_troponine', 'is_bnp',
#     'is_ckmb', 'is_ddimer', 'is_crp', 'is_pct', 'is_alat',
#     'is_asat', 'is_bili_total', 'is_lipase', 'is_alp', 'is_iron',
#     'is_ferritin', 'is_calcium_ionized', 'is_aXa_aIIa', 'is_fibrinogen'
# ]
#
# cols_special_bio = [
#     'has_blood_gas',
#     'has_culture',
#     'has_lumbar_puncture',
# ]
#
# cols_imaging = [
#     'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
#     'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
#     'radio_interventional_1', 'nuclear_medicine_1'
# ]
#
# cols_bio_bin = cols_venous + cols_special_bio  # tout pour les visu
#
# cols_quanti     = ['imaging_exam_count', 'bio_exam_count']
# cols_multi_cat  = cols_imaging   # alias for Top3 signature use
#
# # ── SCENARIOS ─────────────────────────────────────────────────────────────────
#
# SCENARIOS = {
#     # Scenario 1: aggregated imaging + biology flags + disposition
#     "advanced_radio_bio_ekg_dispo": [
#         'has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri',
#         'has_radio_interventional', 'has_nuclear_medicine',
#         'has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#         'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#         'imaging_exam_count', 'bio_exam_count',
#     ],
#     # Scenario 2: detailed venous + detailed imaging + disposition
#     "advanced_detailed_radio_bio_ekg_dispo":
#         cols_venous + cols_special_bio + cols_imaging + [
#             'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#             'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#             'imaging_exam_count', 'bio_exam_count',
#         ]
# }
#
#
# # ============================================================
# # CELL 3 — TEXT BUILDING CONFIG & HELPER FUNCTIONS
# # ============================================================
#
# # ── CONFIGS PER DATASET ───────────────────────────────────────────────────────
#
# CONFIG_DETAILED = {
#     "imaging_mode":    "categorical",        # string-valued imaging columns
#     "imaging_cols":    IMAGING_COLS_DETAILED,
#     "bio_cols":        BIO_COLS,
#     "procedure_cols":  PROCEDURE_COLS,
#     "has_disposition": True,
# }
#
# CONFIG_SIMPLE = {
#     "imaging_mode":    "boolean",            # binary has_* imaging columns
#     "imaging_cols":    IMAGING_COLS_BOOL,
#     "bio_cols":        BIO_COLS,
#     "procedure_cols":  PROCEDURE_COLS,
#     "has_disposition": True,
# }
#
# # ── HELPERS ───────────────────────────────────────────────────────────────────
#
# def _is_present(val) -> bool:
#     """Returns True if val represents a positive/present finding."""
#     if val is None:
#         return False
#     try:
#         if pd.isna(val):
#             return False
#     except Exception:
#         pass
#     if str(val).upper() in ["NONE", "NAN", "", "0", "FALSE"]:
#         return False
#     return val in (1, 1.0, True) or isinstance(val, str)
#
#
# def _clean_label(val):
#     """Normalises a raw categorical value to a readable string, or None."""
#     if val is None:
#         return None
#     try:
#         if pd.isna(val):
#             return None
#     except Exception:
#         pass
#     val = str(val).strip()
#     if val.upper() in ["NONE", "NAN", "", "0"]:
#         return None
#     return val.lower().replace("_", " ")
#
#
# def _join_list(items: list) -> str:
#     """['a', 'b', 'c'] -> 'a, b and c'"""
#     if not items:
#         return ""
#     if len(items) == 1:
#         return items[0]
#     return ", ".join(items[:-1]) + f" and {items[-1]}"
#
#
# def _with_article(s: str) -> str:
#     """Prepends 'an' or 'a' based on first letter."""
#     return f"an {s}" if s[0] in "aeiouAEIOU" else f"a {s}"
#
#
# # ── IMAGING SENTENCE BUILDERS ─────────────────────────────────────────────────
#
# def _imaging_sentence_boolean(row, imaging_cols: dict) -> str | None:
#     """Builds imaging sentence from boolean has_* columns."""
#     done = [label for col, label in imaging_cols.items() if _is_present(row.get(col))]
#     if not done:
#         return None
#     return f"The patient underwent {_join_list(done)}."
#
#
# def _imaging_sentence_categorical(row, imaging_cols: dict) -> str | None:
#     """Builds imaging sentence from categorical string columns."""
#     exams = []
#     for modality, cols in imaging_cols.items():
#         for col in cols:
#             val = _clean_label(row.get(col))
#             if val is not None:
#                 exams.append(_with_article(val))
#     if not exams:
#         return None
#     return f"The patient underwent {_join_list(exams)}."
#
#
# # ── MAIN TEXT BUILDER ─────────────────────────────────────────────────────────
#
# def build_clinical_text(row, config: dict) -> str:
#     sentences = []
#
#     # ── 1. IMAGING ───────────────────────────────────────────────
#     if config["imaging_mode"] == "categorical":
#         imaging_sentence = _imaging_sentence_categorical(row, config["imaging_cols"])
#     else:
#         imaging_sentence = _imaging_sentence_boolean(row, config["imaging_cols"])
#
#     imaging_count = int(row.get('imaging_exam_count') or 0)
#
#     if imaging_sentence:
#         sentences.append(imaging_sentence)
#     elif imaging_count > 0:
#         sentences.append(f"The patient underwent {imaging_count} imaging exam(s).")
#
#     # ── 2. BIOLOGICAL EXAMS ──────────────────────────────────────
#     bio_done  = [label for col, label in config["bio_cols"].items() if _is_present(row.get(col))]
#     bio_count = int(row.get('bio_exam_count') or 0)
#
#     if bio_done:
#         verb = "were" if len(bio_done) > 1 else "was"
#         sentences.append(f"{_join_list(bio_done).capitalize()} {verb} measured.")
#     elif bio_count > 0:
#         sentences.append(f"{bio_count} biological exam(s) were performed.")
#
#     # ── 3. EXAM COUNT SUMMARY ────────────────────────────────────
#     count_parts = []
#     if imaging_count > 0:
#         count_parts.append(f"{imaging_count} imaging exam(s)")
#     if bio_count > 0:
#         count_parts.append(f"{bio_count} biological exam(s)")
#     if count_parts:
#         sentences.append(f"In total, the patient had {_join_list(count_parts)}.")
#
#     # ── 4. PROCEDURES ────────────────────────────────────────────
#     procedures = [label for col, label in config["procedure_cols"].items() if _is_present(row.get(col))]
#     if procedures:
#         sentences.append(f"The patient had {_join_list(procedures)} performed.")
#
#     # ── 5. DISPOSITION ───────────────────────────────────────────
#     if config.get("has_disposition"):
#         observation  = _is_present(row.get('observation_unit'))
#         hospitalized = _is_present(row.get('hospitalization'))
#         transferred  = _is_present(row.get('inter_facility_transfer'))
#
#         hosp_val = row.get('hospitalization')
#         try:
#             discharged = (not pd.isna(hosp_val)) and (hosp_val == 0)
#         except Exception:
#             discharged = False
#
#         if observation and hospitalized:
#             sentences.append("The patient was placed in the observation unit and then admitted to the hospital.")
#         elif observation and transferred:
#             sentences.append("The patient was placed in the observation unit and then transferred to another facility.")
#         elif observation:
#             sentences.append("The patient was placed in the observation unit.")
#         elif hospitalized:
#             sentences.append("The patient was admitted to the hospital.")
#         elif transferred:
#             sentences.append("The patient was transferred to another facility.")
#         elif discharged:
#             sentences.append("The patient was discharged.")
#
#     return " ".join(sentences) if sentences else "No significant resource utilization was documented."
#
#
#
#


In [60]:
# # ============================================================
# # CELL 4 — APPLY TEXT BUILDER TO DATAFRAMES
# # ============================================================
# df = pd.read_csv(CSV_PATH)
# log.info(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
#
# # ── BUILD CLINICAL TEXT ───────────────────────────────────────
# # Choose the config that matches your dataset:
# #   - CONFIG_SIMPLE   : boolean imaging columns (has_ct_scan, etc.)
# #   - CONFIG_DETAILED : categorical imaging columns (ct_scan_1, etc.)
#
# ACTIVE_CONFIG = CONFIG_DETAILED
#
# df['clinical_text'] = df.apply(
#     lambda row: build_clinical_text(row, ACTIVE_CONFIG), axis=1
# )
#
# log.info(f"Clinical text built — example:\n{df['clinical_text'].iloc[0]}")

In [61]:
# # ============================================================
# # CELL 1 — IMPORTS, PATHS & HYPERPARAMETERS
# # ============================================================
#
# import os
# import warnings
# import logging
# import pandas as pd
# import numpy as np
# import torch
# import umap
# import hdbscan
# import matplotlib.pyplot as plt
# import seaborn as sns
# from tqdm import tqdm
# from wordcloud import WordCloud
# from transformers import AutoTokenizer, AutoModel
# from bertopic import BERTopic
# from bertopic.vectorizers import ClassTfidfTransformer
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.metrics import silhouette_score
#
# # pour sentence transfor;er
# from sentence_transformers import SentenceTransformer
#
# warnings.filterwarnings("ignore")
#
# # ── LOGGER ────────────────────────────────────────────────────────────────────
# log = logging.getLogger(__name__)
# logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
#
# # ── THREADING & DEVICE ────────────────────────────────────────────────────────
# os.environ.update({k: "128" for k in [
#     "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
#     "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"
# ]})
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# log.info(f"Device : {device}")
#
# # ── PATHS & MODEL ─────────────────────────────────────────────────────────────
# CSV_PATH   = "df_final_binaire_imputed.csv"
# OUTPUT_DIR = "Results/BERTopic"
# #MODEL_NAME = "yikuan8/Clinical-Longformer"
# MODEL_NAME = SentenceTransformer("sentence-transformers/embeddinggemma-300m-medical")
#
# # ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
# BATCH_SIZE  = 128    # Batch size for embedding computation
# MAX_LENGTH  = 512    # Max token length for Clinical-Longformer
#
# UMAP_DIM       = 10   # UMAP output dimensions for clustering
# UMAP_NEIGHBORS = 15   # n_neighbors for clustering UMAP
# UMAP_MIN_DIST  = 0.0  # min_dist for clustering UMAP (0.0 = tighter clusters)
#
# MCS_VALUES  = range(200, 2100, 200)  # HDBSCAN min_cluster_size sweep values
# MIN_SAMPLES = 10                     # HDBSCAN min_samples
#
# # ── COLUMN DEFINITIONS ────────────────────────────────────────────────────────
# # Binary columns for blood/lab tests
# cols_venous = [
#     'is_hemoglobine', 'is_leucocytes', 'is_formule_leuco', 'is_urea', 'is_creatinine',
#     'is_sodium', 'is_potassium', 'is_platelets', 'is_pt', 'is_aptt',
#     'is_calcium', 'is_ck', 'is_lactates', 'is_troponine', 'is_bnp',
#     'is_ckmb', 'is_ddimer', 'is_crp', 'is_pct', 'is_alat',
#     'is_asat', 'is_bili_total', 'is_lipase', 'is_alp', 'is_iron',
#     'is_ferritin', 'is_calcium_ionized', 'is_aXa_aIIa', 'is_fibrinogen'
# ]
#
# # Categorical imaging columns (multi-category)
# cols_imaging = [
#     'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
#     'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
#     'radio_interventional_1', 'nuclear_medicine_1'
# ]
#
# # Binary biology columns used for heatmap/radar visualization
# cols_bio_bin = [
#     'is_blood_gas', 'is_aXa_aIIa', 'is_csf', 'is_lactates', 'is_culture',
#     'is_leucocytes', 'is_formule_leuco', 'is_alat', 'is_asat', 'is_bnp',
#     'is_bili_total', 'is_ck', 'is_ckmb', 'is_crp', 'is_calcium_ionized',
#     'is_calcium', 'is_creatinine', 'is_ddimer', 'is_iron', 'is_ferritin',
#     'is_fibrinogen', 'is_hemoglobine', 'is_lipase', 'is_alp', 'is_platelets',
#     'is_potassium', 'is_pct', 'is_sodium', 'is_aptt', 'is_pt',
#     'is_troponine', 'is_urea', 'had_ekg'
# ]
#
# # Multi-category columns used for Top3 signature (imaging types)
# cols_multi_cat = cols_imaging
#
# # Quantitative columns (counts)
# cols_quanti = ['imaging_exam_count', 'bio_exam_count']
#
# # ── SCENARIOS ─────────────────────────────────────────────────────────────────
# # Each scenario defines which columns are used to build the clinical text
# SCENARIOS = {
#     # Scenario 1 : aggregated imaging + biology flags + disposition
#     "advanced_radio_bio_ekg_dispo": [
#         'has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri',
#         'has_radio_interventional', 'has_nuclear_medicine',
#         'has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#         'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#         'imaging_exam_count', 'bio_exam_count',
#     ],
#     # Scenario 2 : detailed venous + detailed imaging + disposition
#     "advanced_detailed_radio_bio_ekg_dispo":
#         cols_venous + cols_imaging + [
#             'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#             'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#             'imaging_exam_count', 'bio_exam_count',
#         ]
# }
#
# # ── VERB MAP ──────────────────────────────────────────────────────────────────
# # # Maps binary column names to human-readable clinical sentences
# # VERB_MAP = {
# #     'has_ultrasound':           "underwent ultrasound",
# #     'has_ct_scan':              "underwent CT scan",
# #     'has_xray':                 "underwent X-ray",
# #     'has_mri':                  "underwent MRI",
# #     'has_radio_interventional': "underwent interventional radiology procedure",
# #     'has_nuclear_medicine':     "underwent nuclear medicine imaging",
# #     'has_blood_test':           "had blood work ordered",
# #     'has_culture':              "had microbiological cultures taken",
# #     'has_lumbar_puncture':      "underwent lumbar puncture",
# #     'has_blood_gas':            "had arterial blood gas analysis",
# #     'had_ekg':                  "had an EKG performed",
# #     'hospitalization':          "was admitted to the hospital",
# #     'observation_unit':         "was placed in observation unit",
# #     'inter_facility_transfer':  "was transferred to another facility",
# #     'is_hemoglobine':           "hemoglobin was measured",
# #     'is_leucocytes':            "white blood cell count was obtained",
# #     'is_formule_leuco':         "differential leukocyte count was performed",
# #     'is_urea':                  "blood urea was measured",
# #     'is_creatinine':            "creatinine was assessed",
# #     'is_sodium':                "sodium level was checked",
# #     'is_potassium':             "potassium level was checked",
# #     'is_platelets':             "platelet count was obtained",
# #     'is_pt':                    "prothrombin time was measured",
# #     'is_aptt':                  "aPTT was measured",
# #     'is_calcium':               "calcium level was assessed",
# #     'is_ck':                    "CK was measured",
# #     'is_lactates':              "lactate level was obtained",
# #     'is_troponine':             "troponin was measured",
# #     'is_bnp':                   "BNP was assessed",
# #     'is_ckmb':                  "CK-MB was measured",
# #     'is_ddimer':                "D-dimer was obtained",
# #     'is_crp':                   "CRP was measured",
# #     'is_pct':                   "procalcitonin was assessed",
# #     'is_alat':                  "ALT was measured",
# #     'is_asat':                  "AST was measured",
# #     'is_bili_total':            "total bilirubin was assessed",
# #     'is_lipase':                "lipase was measured",
# #     'is_alp':                   "ALP was measured",
# #     'is_iron':                  "serum iron was assessed",
# #     'is_ferritin':              "ferritin was measured",
# #     'is_calcium_ionized':       "ionized calcium was checked",
# #     'is_aXa_aIIa':              "anti-Xa/anti-IIa activity was measured",
# #     'is_fibrinogen':            "fibrinogen was assessed",
#
#
# IMAGING_COLS = {
#     'has_ultrasound':           "an ultrasound",
#     'has_ct_scan':              "a CT scan",
#     'has_xray':                 "an X-ray",
#     'has_mri':                  "an MRI",
#     'has_radio_interventional': "an interventional radiology procedure",
#     'has_nuclear_medicine':     "a nuclear medicine imaging",
# }
#
# BIO_COLS = {
#     'is_hemoglobine':       "hemoglobin",
#     'is_leucocytes':        "white blood cell count",
#     'is_formule_leuco':     "differential leukocyte count",
#     'is_urea':              "blood urea",
#     'is_creatinine':        "creatinine",
#     'is_sodium':            "sodium",
#     'is_potassium':         "potassium",
#     'is_platelets':         "platelet count",
#     'is_pt':                "prothrombin time",
#     'is_aptt':              "aPTT",
#     'is_calcium':           "calcium",
#     'is_ck':                "CK",
#     'is_lactates':          "lactate",
#     'is_troponine':         "troponin",
#     'is_bnp':               "BNP",
#     'is_ckmb':              "CK-MB",
#     'is_ddimer':            "D-dimer",
#     'is_crp':               "CRP",
#     'is_pct':               "procalcitonin",
#     'is_alat':              "ALT",
#     'is_asat':              "AST",
#     'is_bili_total':        "total bilirubin",
#     'is_lipase':            "lipase",
#     'is_alp':               "ALP",
#     'is_iron':              "serum iron",
#     'is_ferritin':          "ferritin",
#     'is_calcium_ionized':   "ionized calcium",
#     'is_aXa_aIIa':          "anti-Xa/anti-IIa activity",
#     'is_fibrinogen':        "fibrinogen",
#     'has_blood_test':       "blood work",
#     'has_culture':          "microbiological cultures",
#     'has_lumbar_puncture':  "lumbar puncture",
#     'has_blood_gas':        "arterial blood gas",
# }
#
# PROCEDURE_COLS = {
#     'had_ekg': "an EKG",
# }
#


In [62]:
# # # ============================================================
# # # CELL 2 — HELPER FUNCTIONS
# # # ============================================================
# #
# # # ── CONFIGS PAR DATASET ──────────────────────────────────────────
# #
# # CONFIG_DETAILED = {
# #     "imaging_mode": "categorical",  # colonnes avec valeurs string
# #     "imaging_cols": {
# #         'ultrasound':           ['ultrasound_1', 'ultrasound_2'],
# #         'ct_scan':              ['ct_scan_1', 'ct_scan_2', 'ct_scan_3'],
# #         'xray':                 ['xray_1', 'xray_2', 'xray_3'],
# #         'mri':                  ['mri_1', 'mri_2'],
# #         'radio_interventional': ['radio_interventional_1'],
# #         'nuclear_medicine':     ['nuclear_medicine_1'],
# #     },
# #     "bio_cols": BIO_COLS,           # identique dans les 2 datasets ?
# #     "procedure_cols": PROCEDURE_COLS,
# #     "has_disposition": True,
# # }
# #
# # CONFIG_SIMPLE = {
# #     "imaging_mode": "boolean",      # colonnes booléennes has_ct_scan etc.
# #     "imaging_cols": IMAGING_COLS,   # le dict {col: label} d'avant
# #     "bio_cols": BIO_COLS,
# #     "procedure_cols": PROCEDURE_COLS,
# #     "has_disposition": True,
# # }
# #
# #
# #
# #
# #
# #
# #
# #
# #
# #
# #
# # def _clean_exam_label(val):
# #     if val is None:
# #         return None
# #     try:
# #         if pd.isna(val):
# #             return None
# #     except Exception:
# #         pass
# #     val = str(val).strip()
# #     if val.upper() in ["NONE", "NAN", "", "0"]:
# #         return None
# #     return val.lower().replace("_", " ")
# #
# #
# # def build_imaging_sentence(row, imaging_cols_detailed):
# #     """
# #     imaging_cols_detailed : dict {modality: [col1, col2, ...]}
# #     Fonctionne que les valeurs soient 'chest', 'ct chest', etc.
# #     """
# #     all_exams = []
# #
# #     for modality, cols in imaging_cols_detailed.items():
# #         for col in cols:
# #             val = _clean_exam_label(row.get(col))
# #             if val is None:
# #                 continue
# #             all_exams.append(val)  # on prend la valeur telle quelle
# #
# #     if not all_exams:
# #         return None
# #
# #     def with_article(s):
# #         return f"an {s}" if s[0] in "aeiouAEIOU" else f"a {s}"
# #
# #     return f"The patient underwent {_join_list([with_article(e) for e in all_exams])}."
# #
# #
# # # ── FONCTION UNIQUE ──────────────────────────────────────────────
# #
# # def build_clinical_text(row, config):
# #     sentences = []
# #
# #     # ── 1. IMAGING ───────────────────────────────────────────────
# #     if config["imaging_mode"] == "categorical":
# #         imaging_sentence = build_imaging_sentence(row, config["imaging_cols"])
# #     else:
# #         # mode booléen : {col: label}
# #         imaging_done = [label for col, label in config["imaging_cols"].items()
# #                         if _is_present(row.get(col))]
# #         def with_article(s):
# #             return f"an {s}" if s[0] in "aeiouAEIOU" else f"a {s}"
# #         imaging_sentence = (
# #             f"The patient underwent {_join_list([with_article(l) for l in imaging_done])}."
# #             if imaging_done else None
# #         )
# #
# #     imaging_count = row.get('imaging_exam_count', 0) or 0
# #     if imaging_sentence:
# #         sentences.append(imaging_sentence)
# #     elif imaging_count > 0:
# #         sentences.append(f"The patient underwent {int(imaging_count)} imaging exam(s).")
# #
# #     # ── 2. BIOLOGICAL EXAMS ──────────────────────────────────────
# #     bio_done = [label for col, label in config["bio_cols"].items()
# #                 if _is_present(row.get(col))]
# #     bio_count = row.get('bio_exam_count', 0) or 0
# #
# #     if bio_done:
# #         verb = "were" if len(bio_done) > 1 else "was"
# #         sentences.append(f"{_join_list(bio_done).capitalize()} {verb} measured.")
# #     elif bio_count > 0:
# #         sentences.append(f"{int(bio_count)} biological exam(s) were performed.")
# #
# #     # ── 3. SUMMARY COUNTS ────────────────────────────────────────
# #     count_parts = []
# #     if imaging_count > 0:
# #         count_parts.append(f"{int(imaging_count)} imaging exam(s)")
# #     if bio_count > 0:
# #         count_parts.append(f"{int(bio_count)} biological exam(s)")
# #     if count_parts:
# #         sentences.append(f"In total, the patient had {_join_list(count_parts)}.")
# #
# #     # ── 4. PROCEDURES ────────────────────────────────────────────
# #     procedures = [label for col, label in config["procedure_cols"].items()
# #                   if _is_present(row.get(col))]
# #     if procedures:
# #         sentences.append(f"The patient had {_join_list(procedures)} performed.")
# #
# #     # ── 5. DISPOSITION ───────────────────────────────────────────
# #     if config.get("has_disposition"):
# #         observation  = _is_present(row.get('observation_unit'))
# #         hospitalized = _is_present(row.get('hospitalization'))
# #         transferred  = _is_present(row.get('inter_facility_transfer'))
# #         hosp_val     = row.get('hospitalization', None)
# #         try:
# #             discharged = (not pd.isna(hosp_val)) and (hosp_val == 0)
# #         except Exception:
# #             discharged = False
# #
# #         if observation and hospitalized:
# #             sentences.append("The patient was placed in the observation unit and then admitted to the hospital.")
# #         elif observation and transferred:
# #             sentences.append("The patient was placed in the observation unit and then transferred to another facility.")
# #         elif observation:
# #             sentences.append("The patient was placed in the observation unit.")
# #         elif hospitalized:
# #             sentences.append("The patient was admitted to the hospital.")
# #         elif transferred:
# #             sentences.append("The patient was transferred to another facility.")
# #         elif discharged:
# #             sentences.append("The patient was discharged.")
# #
# #     return " ".join(sentences) if sentences else "No significant resource utilization was documented."
# #
# #
# # # ── USAGE ────────────────────────────────────────────────────────
# #
# # df_detailed['clinical_text'] = df_detailed.apply(
# #     lambda row: build_clinical_text(row, CONFIG_DETAILED), axis=1
# # )
# #
# # df_simple['clinical_text'] = df_simple.apply(
# #     lambda row: build_clinical_text(row, CONFIG_SIMPLE), axis=1
# # )
#
#
# # ## Résumé de l'approche
# # ```
# # CONFIG_DETAILED  ──┐
# #                    ├──▶  build_clinical_text(row, config)
# # CONFIG_SIMPLE    ──┘
#
#
#
# @torch.no_grad()
# def compute_embeddings(texts, tokenizer, model):
#     """
#     Generates sentence embeddings using mean pooling over token embeddings.
#     Processes texts in batches to avoid memory overflow.
#     Returns a numpy array of shape (n_texts, hidden_size).
#     """
#     all_embs = []
#     model.eval()
#     for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding"):
#         batch  = texts[i : i + BATCH_SIZE]
#         inputs = tokenizer(
#             batch, padding=True, truncation=True,
#             max_length=MAX_LENGTH, return_tensors="pt"
#         ).to(device)
#         out  = model(**inputs)
#
#         # Mean pooling : average token embeddings weighted by attention mask
#         mask = inputs["attention_mask"].unsqueeze(-1).float()
#         emb  = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
#         all_embs.append(emb.cpu().numpy())
#
#     return np.vstack(all_embs)
#
#
# def calculate_metrics(umap_embeddings, labels, hdbscan_model):
#     """
#     Computes clustering quality metrics:
#     - Silhouette score  : separation between clusters (higher = better)
#     - Stability         : mean HDBSCAN cluster persistence (higher = denser)
#     - Outlier rate      : fraction of points labelled as noise (-1)
#     - Combined score    : weighted combination of the three metrics
#
#     Returns (silhouette, stability, outlier_rate, combined_score).
#     Returns zeros if clustering is degenerate (< 2 clusters).
#     """
#     mask       = labels != -1
#     n_valid    = np.sum(mask)
#     n_clusters = len(np.unique(labels[mask]))
#
#     # Guard : silhouette is undefined with fewer than 2 clusters
#     if n_valid < 2 or n_clusters < 2:
#         log.warning("⚠️ Degenerate clustering — returning zero scores")
#         return 0.0, 0.0, 1.0, 0.0
#
#     silhouette   = silhouette_score(umap_embeddings[mask], labels[mask], sample_size=3000)
#     outlier_rate = np.sum(labels == -1) / len(labels)
#     stability    = np.mean(hdbscan_model.cluster_persistence_)
#
#     # Weighted combined score (all terms in [0, 1])
#     combined = (silhouette * 0.4) + (stability * 0.4) + ((1 - outlier_rate) * 0.2)
#
#     return silhouette, stability, outlier_rate, combined

In [63]:
# # ============================================================
# # CELL 3 — PIPELINE 2 : EMBEDDING + UMAP + HDBSCAN SWEEP
# # ============================================================
#
# # Load data and model
# df        = pd.read_csv(CSV_PATH, low_memory=False)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model     = AutoModel.from_pretrained(MODEL_NAME).to(device)
# log.info(f"✅ Data loaded : {df.shape} | Model : {MODEL_NAME}")
#
# for sc_name, sc_cols in SCENARIOS.items():
#     log.info(f"\n{'='*60}\nSCENARIO : {sc_name}\n{'='*60}")
#
#     sc_dir = os.path.join(OUTPUT_DIR, sc_name)
#     os.makedirs(sc_dir, exist_ok=True)
#
#     # ── STEP A : Build clinical text documents ────────────────────────────────
#     # Each patient row is converted into a clinical sentence
#     docs = [build_clinical_text(row, sc_cols) for _, row in df.iterrows()]
#     log.info(f"📝 {len(docs)} clinical documents built")
#
#     # ── STEP B : Compute embeddings (done once per scenario) ──────────────────
#     # Heavy step — Clinical-Longformer encodes each document into a vector
#     embs = compute_embeddings(docs, tokenizer, model)
#     sc_dir = os.path.join(OUTPUT_DIR, sc_name)
#     os.makedirs(sc_dir, exist_ok=True)
#     np.save(os.path.join(sc_dir, f"{sc_name}_embeddings.npy"), embs)
#     log.info(f"💾 Raw embeddings saved : {embs.shape}")
#
#     # ── STEP C : UMAP dimensionality reduction (for clustering) ───────────────
#     # Reduces high-dim embeddings to UMAP_DIM dimensions
#     # cosine metric is appropriate for high-dimensional sentence embeddings
#     log.info(f"🔻 UMAP reduction → {UMAP_DIM}D (cosine)...")
#     umap_model    = umap.UMAP(
#         n_neighbors=UMAP_NEIGHBORS,
#         n_components=UMAP_DIM,
#         min_dist=UMAP_MIN_DIST,
#         metric="cosine",
#         random_state=42
#     )
#     umap_embeddings = umap_model.fit_transform(embs)
#
#     # Save UMAP embeddings — reused by Pipeline 3 for visualization
#     np.save(os.path.join(sc_dir, f"{sc_name}_umap_embeddings.npy"), umap_embeddings)
#     log.info(f"💾 UMAP embeddings saved : {umap_embeddings.shape}")
#
#     sweep_results = []
#
#     # ── STEP D : HDBSCAN sweep over min_cluster_size values ──────────────────
#     for mcs in MCS_VALUES:
#         log.info(f"--- MCS = {mcs} ---")
#         mcs_dir = os.path.join(sc_dir, f"mcs_{mcs}")
#         os.makedirs(mcs_dir, exist_ok=True)
#
#         # 1. HDBSCAN clustering on UMAP embeddings (euclidean in reduced space)
#         hdbscan_model = hdbscan.HDBSCAN(
#             min_cluster_size=mcs,
#             min_samples=MIN_SAMPLES,
#             metric="euclidean",         # euclidean is appropriate after UMAP reduction
#             gen_min_span_tree=True      # needed for cluster_persistence_
#         )
#         labels  = hdbscan_model.fit_predict(umap_embeddings)
#         n_clust = len(np.unique(labels)) - 1  # exclude noise label (-1)
#         log.info(f"   → {n_clust} clusters | {np.sum(labels == -1)} noise points")
#
#         # 2. BERTopic for topic interpretation of each cluster
#         # We reuse the pre-computed HDBSCAN model and raw embeddings
#         topic_model = BERTopic(
#             embedding_model=None,           # embeddings already computed
#             hdbscan_model=hdbscan_model,
#             vectorizer_model=CountVectorizer(
#                 token_pattern=r"(?u)\b\w[\w_]+\b",
#                 stop_words="english"
#             ),
#             ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
#         )
#         topics, _ = topic_model.fit_transform(docs, embs)
#
#         # 3. Compute quality metrics
#         silhouette, stability, outlier_rate, combined = calculate_metrics(
#             umap_embeddings, labels, hdbscan_model
#         )
#
#         sweep_results.append({
#             "mcs":          mcs,
#             "n_clusters":   n_clust,
#             "silhouette":   silhouette,
#             "stability":    stability,
#             "outlier_rate": outlier_rate,
#             "combined_score": combined
#         })
#
#         # 4. Save local results for this MCS
#         topic_model.get_topic_info().to_csv(
#             os.path.join(mcs_dir, "topics_summary.csv"), index=False
#         )
#         topic_model.visualize_barchart(top_n_topics=15).write_html(
#             os.path.join(mcs_dir, "viz_clusters_words.html")
#         )
#
#         # 5. UMAP 2D control plot for the first MCS only
#         # euclidean metric because we project already-reduced UMAP embeddings
#         if mcs == MCS_VALUES[0]:
#             log.info("   🗺️ Generating UMAP 2D control plot...")
#             umap_ctrl = umap.UMAP(
#                 n_components=2, n_neighbors=15,
#                 min_dist=0.1, metric="euclidean", random_state=42
#             ).fit_transform(umap_embeddings)
#
#             plt.figure(figsize=(10, 7))
#             sns.scatterplot(
#                 x=umap_ctrl[:, 0], y=umap_ctrl[:, 1],
#                 hue=labels, palette="viridis", s=5, alpha=0.6
#             )
#             plt.title(f"UMAP 2D Control : {sc_name} | MCS={mcs}")
#             plt.savefig(os.path.join(mcs_dir, "umap_control_view.png"))
#             plt.close()
#
#     # ── STEP E : Sweep summary ────────────────────────────────────────────────
#     df_sweep = pd.DataFrame(sweep_results)
#     df_sweep.to_csv(os.path.join(sc_dir, "sweep_master_report.csv"), index=False)
#
#     # Find best MCS based on combined score
#     best_mcs = df_sweep.loc[df_sweep["combined_score"].idxmax(), "mcs"]
#     log.info(f"\n⭐ Best MCS for {sc_name} : {best_mcs}")
#
#     # ── STEP F : Performance curves ───────────────────────────────────────────
#     plt.figure(figsize=(12, 7))
#     plt.plot(df_sweep["mcs"], df_sweep["combined_score"], "k-o", label="Combined Score", linewidth=3)
#     plt.plot(df_sweep["mcs"], df_sweep["silhouette"],     "r--", label="Silhouette")
#     plt.plot(df_sweep["mcs"], df_sweep["stability"],      "b--", label="Stability")
#     plt.axvline(x=best_mcs, color="orange", linestyle=":", label=f"Best MCS ({best_mcs})")
#     plt.title(f"Clustering Optimization : {sc_name}")
#     plt.xlabel("Min Cluster Size (MCS)")
#     plt.ylabel("Scores")
#     plt.legend()
#     plt.grid(True)
#     plt.savefig(os.path.join(sc_dir, "performance_curves.png"))
#     plt.show()
#
# log.info("\n✅ Pipeline 2 complete — results saved in Results/BERTopic/")

In [64]:
# # ============================================================
# # CELL 4 — PIPELINE 3 : FINAL VISUALIZATIONS
# # ============================================================
#
# def run_visualization():
#     for sc_name in SCENARIOS:
#         sc_dir  = os.path.join(OUTPUT_DIR, sc_name)
#         out_dir = os.path.join(sc_dir, "cluster_sweep")
#
#         # ── Load UMAP embeddings and best cluster labels ──────────────────────
#         # umap_embeddings : reduced embeddings used during clustering
#         # best_labels     : best cluster assignment from sweep
#         umap_emb_path = os.path.join(sc_dir, f"{sc_name}_umap_embeddings.npy")
#         labels_path   = os.path.join(out_dir, f"{sc_name}_best_labels.npy")
#
#         if not os.path.exists(umap_emb_path) or not os.path.exists(labels_path):
#             log.warning(f"❌ Missing files for {sc_name}, skipping")
#             continue
#
#         log.info(f"\n🚀 VISUALIZATION : {sc_name}")
#         umap_embeddings = np.load(umap_emb_path)
#         best_labels     = np.load(labels_path)
#         log.info(f"📊 UMAP embeddings shape : {umap_embeddings.shape} | N points : {len(best_labels)}")
#
#         # ── Reconstruct df_valid with clinical text ───────────────────────────
#         # Assign cluster labels and filter out noise points (-1)
#         df_sc            = df.copy()
#         df_sc["cluster"] = best_labels
#         valid_cols       = [c for c in SCENARIOS[sc_name] if c in df_sc.columns]
#         df_sc["text"]    = df_sc.apply(lambda row: build_clinical_text(row, valid_cols), axis=1)
#         df_valid         = df_sc[df_sc["cluster"] != -1].copy()
#         log.info(f"   → {len(df_valid)} valid patients (noise excluded)")
#
#         # ── Available column subsets for this scenario ────────────────────────
#         c_bio    = [c for c in cols_bio_bin   if c in df_valid.columns]
#         c_multi  = [c for c in cols_multi_cat if c in df_valid.columns]
#         c_quanti = [c for c in cols_quanti    if c in df_valid.columns]
#
#         # ── Wordclouds : most frequent terms per cluster ──────────────────────
#         plot_wordclouds(df_valid, sc_name, out_dir)
#
#         # ── Radar chart : binary bio profile per cluster ──────────────────────
#         if c_bio:
#             plot_radar(df_valid, c_bio, sc_name, out_dir)
#
#         # ── Heatmap : binary bio frequency per cluster ────────────────────────
#         if c_bio:
#             plot_heatmap(df_valid, c_bio, sc_name, out_dir)
#
#         # ── UMAP 2D projections for cluster visualization ─────────────────────
#         # We project the UMAP embeddings (already reduced) into 2D
#         # euclidean metric is correct here (cosine is for raw high-dim embeddings)
#
#         # Global view : large n_neighbors captures overall structure
#         log.info("🗺️ UMAP 2D — global view (n_neighbors=30)...")
#         umap_global = umap.UMAP(
#             n_components=2, n_neighbors=30,
#             min_dist=0.1, metric="euclidean", random_state=42
#         ).fit_transform(umap_embeddings)
#
#         # Local view : small n_neighbors highlights fine cluster separation
#         log.info("🗺️ UMAP 2D — local view (n_neighbors=10)...")
#         umap_local = umap.UMAP(
#             n_components=2, n_neighbors=10,
#             min_dist=0.01, metric="euclidean", random_state=42
#         ).fit_transform(umap_embeddings)
#
#         # ── Plot both projections side by side ────────────────────────────────
#         best_config = pd.read_csv(os.path.join(out_dir, f"{sc_name}_best_config.csv")).iloc[0]
#
#         fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
#         title = (
#             f"BEST MODEL : {sc_name.upper()}\n"
#             f"Clusters: {int(best_config['n_clusters'])} | "
#             f"MCS: {int(best_config['mcs'])} | "
#             f"Silhouette: {best_config['silhouette']:.3f} | "
#             f"Stability: {best_config['stability']:.3f} | "
#             f"Outliers: {best_config['outlier_rate']*100:.1f}%"
#         )
#         fig.suptitle(title, fontsize=14, fontweight="bold")
#
#         for ax, coord, name in [
#             (ax1, umap_global, "UMAP Global (n_neighbors=30)"),
#             (ax2, umap_local,  "UMAP Local  (n_neighbors=10)")
#         ]:
#             # Plot noise points in grey
#             ax.scatter(
#                 coord[best_labels == -1, 0], coord[best_labels == -1, 1],
#                 s=1, color="lightgrey", alpha=0.2, label="Noise"
#             )
#             # Plot each cluster with a distinct color
#             for cluster_id in sorted(set(best_labels) - {-1}):
#                 ax.scatter(
#                     coord[best_labels == cluster_id, 0],
#                     coord[best_labels == cluster_id, 1],
#                     s=4, alpha=0.6, label=f"C{cluster_id}"
#                 )
#             ax.set_title(name)
#             ax.legend(markerscale=3, fontsize=8, loc="best")
#
#         plt.tight_layout(rect=[0, 0.03, 1, 0.90])
#         plt.savefig(os.path.join(out_dir, f"{sc_name}_FINAL_OPTIMUM.png"), dpi=150)
#         plt.close()
#         log.info(f"💾 Plot saved : {sc_name}_FINAL_OPTIMUM.png")
#
#         # ── Cluster signatures CSV ────────────────────────────────────────────
#         # Binary bio features : mean % per cluster
#         res_bio = df_valid.groupby("cluster")[c_bio].mean() * 100 if c_bio else pd.DataFrame()
#
#         # Multi-category features : Top 3 most frequent values per cluster
#         list_top3 = []
#         for col in c_multi:
#             ct = pd.crosstab(df_valid["cluster"], df_valid[col], normalize="index") * 100
#             def get_top3(row):
#                 top = row.sort_values(ascending=False).head(3)
#                 return " | ".join([f"{n} ({v:.1f}%)" for n, v in top.items() if v > 0])
#             list_top3.append(pd.DataFrame(ct.apply(get_top3, axis=1), columns=[f"Top3_{col}"]))
#
#         # Quantitative features : mean ± std per cluster
#         quanti_list = []
#         if c_quanti:
#             res_quanti = pd.DataFrame(index=df_valid.groupby("cluster")[c_quanti].mean().index)
#             for col in c_quanti:
#                 m = df_valid.groupby("cluster")[col].mean().round(1)
#                 s = df_valid.groupby("cluster")[col].std().round(1)
#                 res_quanti[f"{col}_mean±std"] = m.astype(str) + " ± " + s.astype(str)
#             quanti_list = [res_quanti]
#
#         # Merge all signature components
#         final_sigs               = pd.concat([res_bio] + list_top3 + quanti_list, axis=1)
#         counts                   = df_valid["cluster"].value_counts()
#         final_sigs["N_patients"] = counts
#         final_sigs["Weight_%"]   = (counts / len(df) * 100).round(1)
#         if c_bio:
#             final_sigs[c_bio]    = final_sigs[c_bio].round(1)
#
#         out_csv = os.path.join(out_dir, f"signatures_{sc_name}.csv")
#         final_sigs.sort_values("N_patients", ascending=False).to_csv(out_csv)
#         log.info(f"💾 Signatures saved : {out_csv}")
#
#     log.info("\n✅ Pipeline 3 complete !")
#
#
# if __name__ == "__main__":
#     run_visualization()

In [65]:
# # # ============================================================
# # # CONFIG — imports, chemins, hyperparamètres
# # # ============================================================
# #
#
#
# import os
# import warnings
# import pandas as pd
# import numpy as np
# import torch
# import umap
# import hdbscan
# from tqdm import tqdm
# from transformers import AutoTokenizer, AutoModel
# from bertopic import BERTopic
# from bertopic.vectorizers import ClassTfidfTransformer
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.metrics import silhouette_score
# import matplotlib.pyplot as plt
# import seaborn as sns
# from wordcloud import WordCloud
# import logging
# log = logging.getLogger(__name__)
# logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
#
# warnings.filterwarnings("ignore")
#
# # ── THREADING & DEVICE ────────────────────────────────────────────────────────
# os.environ.update({k: "128" for k in ["OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"]})
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#
# # ── CHEMINS & MODÈLE ──────────────────────────────────────────────────────────
# CSV_PATH    = "df_final_binaire_imputed.csv"
# OUTPUT_DIR  = "Results/BERTopic"
# MODEL_NAME  = "yikuan8/Clinical-Longformer"
#
# # ── HYPERPARAMÈTRES ───────────────────────────────────────────────────────────
# BATCH_SIZE     = 128
# MAX_LENGTH     = 512
# UMAP_DIM       = 10
# UMAP_NEIGHBORS = 15
# UMAP_MIN_DIST_PRE_CLUSTER = 0.0
#
# # Paramètres pour le SWEEP HDBSCAN
# MCS_VALUES     = range(200, 2100, 200) # Test de 200 à 2000
# MIN_SAMPLES    = 10
#
# # ── SCÉNARIOS ─────────────────────────────────────────────────────────────────
# # # ── COLONNES ──────────────────────────────────────────────────────────────────
# cols_veinous_analysis = [
#     'is_hemoglobine', 'is_leucocytes', 'is_formule_leuco', 'is_urea', 'is_creatinine',
#     'is_sodium', 'is_potassium', 'is_platelets', 'is_pt', 'is_aptt',
#     'is_calcium', 'is_ck', 'is_lactates', 'is_troponine', 'is_bnp',
#     'is_ckmb', 'is_ddimer', 'is_crp', 'is_pct', 'is_alat',
#     'is_asat', 'is_bili_total', 'is_lipase', 'is_alp', 'is_iron',
#     'is_ferritin', 'is_calcium_ionized', 'is_aXa_aIIa', 'is_fibrinogen'
# ]
# cols_imaging = [
#     'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
#     'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
#     'radio_interventional_1', 'nuclear_medicine_1'
# ]
# cols_bio_bin = [
#     'is_blood_gas', 'is_aXa_aIIa', 'is_csf', 'is_lactates', 'is_culture',
#     'is_leucocytes', 'is_formule_leuco', 'is_alat', 'is_asat', 'is_bnp',
#     'is_bili_total', 'is_ck', 'is_ckmb', 'is_crp', 'is_calcium_ionized',
#     'is_calcium', 'is_creatinine', 'is_ddimer', 'is_iron', 'is_ferritin',
#     'is_fibrinogen', 'is_hemoglobine', 'is_lipase', 'is_alp', 'is_platelets',
#     'is_potassium', 'is_pct', 'is_sodium', 'is_aptt', 'is_pt',
#     'is_troponine', 'is_urea', 'had_ekg'
# ]
# cols_multi_cat = [
#     'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
#     'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
#     'radio_interventional_1', 'nuclear_medicine_1'
# ]
# cols_quanti = ['imaging_exam_count', 'bio_exam_count']
#
# # ── SCÉNARIOS ─────────────────────────────────────────────────────────────────
# SCENARIOS = {
#     "advanced_radio_bio_ekg_dispo": [
#         'has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri',
#         'has_radio_interventional', 'has_nuclear_medicine',
#         'has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#         'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#         'imaging_exam_count', 'bio_exam_count',
#     ],
#     "advanced_detailed_radio_bio_ekg_dispo":
#         cols_veinous_analysis + cols_imaging + [
#             'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#             'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#             'imaging_exam_count', 'bio_exam_count',
#         ]
# }
#
# # ── VERB MAP ──────────────────────────────────────────────────────────────────
# VERB_MAP = {
#     'has_ultrasound':           "underwent ultrasound",
#     'has_ct_scan':              "underwent CT scan",
#     'has_xray':                 "underwent X-ray",
#     'has_mri':                  "underwent MRI",
#     'has_radio_interventional': "underwent interventional radiology procedure",
#     'has_nuclear_medicine':     "underwent nuclear medicine imaging",
#     'has_blood_test':           "had blood work ordered",
#     'has_culture':              "had microbiological cultures taken",
#     'has_lumbar_puncture':      "underwent lumbar puncture",
#     'has_blood_gas':            "had arterial blood gas analysis",
#     'had_ekg':                  "had an EKG performed",
#     'hospitalization':          "was admitted to the hospital",
#     'observation_unit':         "was placed in observation unit",
#     'inter_facility_transfer':  "was transferred to another facility",
#     'is_hemoglobine':           "hemoglobin was measured",
#     'is_leucocytes':            "white blood cell count was obtained",
#     'is_formule_leuco':         "differential leukocyte count was performed",
#     'is_urea':                  "blood urea was measured",
#     'is_creatinine':            "creatinine was assessed",
#     'is_sodium':                "sodium level was checked",
#     'is_potassium':             "potassium level was checked",
#     'is_platelets':             "platelet count was obtained",
#     'is_pt':                    "prothrombin time was measured",
#     'is_aptt':                  "aPTT was measured",
#     'is_calcium':               "calcium level was assessed",
#     'is_ck':                    "CK was measured",
#     'is_lactates':              "lactate level was obtained",
#     'is_troponine':             "troponin was measured",
#     'is_bnp':                   "BNP was assessed",
#     'is_ckmb':                  "CK-MB was measured",
#     'is_ddimer':                "D-dimer was obtained",
#     'is_crp':                   "CRP was measured",
#     'is_pct':                   "procalcitonin was assessed",
#     'is_alat':                  "ALT was measured",
#     'is_asat':                  "AST was measured",
#     'is_bili_total':            "total bilirubin was assessed",
#     'is_lipase':                "lipase was measured",
#     'is_alp':                   "ALP was measured",
#     'is_iron':                  "serum iron was assessed",
#     'is_ferritin':              "ferritin was measured",
#     'is_calcium_ionized':       "ionized calcium was checked",
#     'is_aXa_aIIa':              "anti-Xa/anti-IIa activity was measured",
#     'is_fibrinogen':            "fibrinogen was assessed",
# }

In [66]:
# # # ============================================================
# # # FONCTIONS
# # # ============================================================
#
#
#
# def build_clinical_text(row, cols):
#     parts = []
#     for col in cols:
#         val = row[col]
#         if pd.isna(val) or val == 0 or str(val).upper() in ["NONE", "NAN", ""]:
#             continue
#
#         # Correction ici : Utilisation du VERB_MAP
#         if col in VERB_MAP and (val == 1 or val == 1.0):
#             parts.append(VERB_MAP[col].lower().replace(" ", "_"))
#         elif isinstance(val, str):
#             parts.append(val.lower().replace(" ", "_"))
#         elif val == 1 or val == 1.0:
#             parts.append(col.replace('is_','').replace('has_','').replace('had_',''))
#
#     return "patient " + " ".join(parts) if parts else "no_resource_utilization"
#
#
# @torch.no_grad()
# def compute_embeddings(texts, tokenizer, model):
#     """Extraction des embeddings avec Mean Pooling."""
#     all_embs = []
#     model.eval()
#     for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding"):
#         batch = texts[i : i + BATCH_SIZE]
#         inputs = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
#         out = model(**inputs)
#         mask = inputs["attention_mask"].unsqueeze(-1).float()
#         emb = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
#         all_embs.append(emb.cpu().numpy())
#     return np.vstack(all_embs)
#
#
#
# def calculate_metrics(embeddings, labels, hdbscan_obj):
#     # On filtre les outliers (-1) pour la silhouette
#     mask = labels != -1
#     if np.sum(mask) < 2 or len(np.unique(labels[mask])) < 2:
#         return 0.0, 0.0, 1.0, 0.0
#
#     sil = silhouette_score(embeddings[mask], labels[mask], sample_size=3000)
#     outlier_rate = np.sum(labels == -1) / len(labels)
#     # Stabilité moyenne des clusters (plus c'est haut, plus les clusters sont denses)
#     stability = np.mean(hdbscan_obj.cluster_persistence_)
#
#     # Score Combiné (Normalisé entre 0 et 1)
#     combined = (sil * 0.4) + (stability * 0.4) + ((1 - outlier_rate) * 0.2)
#
#     return sil, stability, outlier_rate, combined


In [67]:
# # 1. Initialisation du matériel
# df = pd.read_csv(CSV_PATH, low_memory=False)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModel.from_pretrained(MODEL_NAME).to(device)
#
# # 2. Boucle sur les Scénarios
# for sc_name, sc_cols in SCENARIOS.items():
#     print(f"\n" + "="*50 + f"\nSCÉNARIO : {sc_name}\n" + "="*50)
#
#     sc_dir = os.path.join(OUTPUT_DIR, sc_name)
#     os.makedirs(sc_dir, exist_ok=True)
#
#     # Étape A : On transforme les données en texte clinique
#     docs = [build_clinical_text(row, sc_cols) for _, row in df.iterrows()]
#
#     # Étape B : On génère les vecteurs (Le plus long, fait une seule fois)
#     embs = compute_embeddings(docs, tokenizer, model)
#
#     # ✅ Sauvegarde embeddings bruts
#     np.save(os.path.join(sc_dir, f"{sc_name}_embeddings.npy"), embs)
#     print(f"💾 Embeddings bruts sauvegardés : {embs.shape}")
#
#     # Étape C : Réduction UMAP pour stabiliser le clustering
#     u_mod_clustering = umap.UMAP(
#         n_neighbors=UMAP_NEIGHBORS,
#         n_components=UMAP_DIM,
#         min_dist=UMAP_MIN_DIST_PRE_CLUSTER,
#         metric='cosine',
#         random_state=42
#     )
#     reduced_embs = u_mod_clustering.fit_transform(embs)
#
#     # ✅ Sauvegarde embeddings UMAP réduits (utilisés pour clustering ET visualisation)
#     np.save(os.path.join(sc_dir, f"{sc_name}_umap_embeddings.npy"), reduced_embs)
#     print(f"💾 Embeddings UMAP sauvegardés : {reduced_embs.shape}")
#
#     sweep_results = []
#
#     # Étape D : La boucle de SWEEP (Test de la granularité)
#     for mcs in MCS_VALUES:
#         print(f"--- Analyse avec MCS = {mcs} ---")
#         path = os.path.join(OUTPUT_DIR, sc_name, f"mcs_{mcs}")
#         os.makedirs(path, exist_ok=True)
#
#         # 1. Clustering
#         h_mod = hdbscan.HDBSCAN(
#             min_cluster_size=mcs, min_samples=MIN_SAMPLES,
#             metric='euclidean', gen_min_span_tree=True
#         )
#         labels = h_mod.fit_predict(reduced_embs)
#
#         # 2. BERTopic
#         topic_model = BERTopic(
#             embedding_model=None,
#             hdbscan_model=h_mod,
#             vectorizer_model=CountVectorizer(
#                 token_pattern=r"(?u)\b\w[\w_]+\b", stop_words="english"
#             ),
#             ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
#         )
#         topics, _ = topic_model.fit_transform(docs, embs)
#
#         # 3. Calcul des scores
#         sil, stab, out_rate, comb = calculate_metrics(reduced_embs, labels, h_mod)
#         n_clust = len(np.unique(labels)) - 1
#
#         sweep_results.append({
#             "mcs": mcs, "silhouette": sil, "stability": stab,
#             "outlier_rate": out_rate, "combined_score": comb, "n_clusters": n_clust
#         })
#
#         # 4. Sauvegardes locales
#         topic_model.get_topic_info().to_csv(f"{path}/topics_summary.csv", index=False)
#         topic_model.visualize_barchart(top_n_topics=15).write_html(f"{path}/viz_clusters_words.html")
#
#         # ✅ UMAP de contrôle à la place du t-SNE (plus rapide)
#         if mcs == MCS_VALUES[0]:
#             umap_ctrl = umap.UMAP(
#                 n_components=2, n_neighbors=15,
#                 min_dist=0.1, metric="euclidean", random_state=42
#             ).fit_transform(reduced_embs)
#             plt.figure(figsize=(10, 7))
#             sns.scatterplot(x=umap_ctrl[:,0], y=umap_ctrl[:,1],
#                             hue=labels, palette='viridis', s=5, alpha=0.6)
#             plt.title(f"Visualisation UMAP contrôle : {sc_name} | MCS={mcs}")
#             plt.savefig(f"{path}/umap_control_view.png")
#             plt.close()
#
#     # Étape E : Synthèse du scénario
#     df_sweep = pd.DataFrame(sweep_results)
#     df_sweep.to_csv(f"{OUTPUT_DIR}/{sc_name}/sweep_master_report.csv", index=False)
#
#     best_mcs = df_sweep.loc[df_sweep['combined_score'].idxmax(), 'mcs']
#     print(f"\n-> Meilleur réglage pour {sc_name} : MCS = {best_mcs}")
#
#     # Étape F : Graphique des performances
#     plt.figure(figsize=(12, 7))
#     plt.plot(df_sweep['mcs'], df_sweep['combined_score'], 'k-o', label='Score Combiné', linewidth=3)
#     plt.plot(df_sweep['mcs'], df_sweep['silhouette'],     'r--', label='Silhouette')
#     plt.plot(df_sweep['mcs'], df_sweep['stability'],      'b--', label='Stabilité')
#     plt.axvline(x=best_mcs, color='orange', linestyle=':', label=f'Best MCS ({best_mcs})')
#     plt.title(f"Optimisation du Clustering : {sc_name}")
#     plt.xlabel("Min Cluster Size (MCS)"); plt.ylabel("Scores")
#     plt.legend(); plt.grid(True)
#     plt.savefig(f"{OUTPUT_DIR}/{sc_name}/performance_curves.png")
#     plt.show()
#
# print("\nAnalyse terminée. Tous les résultats sont dans Results/BERTopic/")

In [68]:
# # 1. Initialisation du matériel
# df = pd.read_csv(CSV_PATH, low_memory=False)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModel.from_pretrained(MODEL_NAME).to(device)
#
# # 2. Boucle sur les Scénarios
# for sc_name, sc_cols in SCENARIOS.items():
#     print(f"\n" + "="*50 + f"\nSCÉNARIO : {sc_name}\n" + "="*50)
#
#     # Étape A : On transforme les données en texte clinique
#     docs = [build_clinical_text(row, sc_cols) for _, row in df.iterrows()]
#
#     # Étape B : On génère les vecteurs (Le plus long, fait une seule fois)
#     embs = compute_embeddings(docs, tokenizer, model)
#
#     # Étape C : Réduction UMAP pour stabiliser le clustering
#     u_mod_clustering = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIM, min_dist= UMAP_MIN_DIST_PRE_CLUSTER,
#                                  metric='cosine', random_state=42)
#     reduced_embs = u_mod_clustering.fit_transform(embs)
#
#     sweep_results = []
#
#     # Étape D : La boucle de SWEEP (Test de la granularité)
#     for mcs in MCS_VALUES:
#         print(f"--- Analyse avec MCS = {mcs} ---")
#         path = os.path.join(OUTPUT_DIR, sc_name, f"mcs_{mcs}")
#         os.makedirs(path, exist_ok=True)
#
#         # 1. Clustering
#         h_mod = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=MIN_SAMPLES,
#                                 metric='euclidean', gen_min_span_tree=True)
#         labels = h_mod.fit_predict(reduced_embs)
#
#         # 2. BERTopic (Interprétation des clusters trouvés)
#         topic_model = BERTopic(
#             embedding_model=None, # On utilise les embs déjà calculés
#             hdbscan_model=h_mod,
#             vectorizer_model=CountVectorizer(token_pattern=r"(?u)\b\w[\w_]+\b", stop_words="english"),
#             ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
#         )
#         topics, _ = topic_model.fit_transform(docs, embs)
#
#         # 3. Calcul des scores (Qualité, Stabilité, Bruit)
#         sil, stab, out_rate, comb = calculate_metrics(reduced_embs, labels, h_mod)
#         n_clust = len(np.unique(labels)) - 1
#
#         sweep_results.append({
#             "mcs": mcs, "silhouette": sil, "stability": stab,
#             "outlier_rate": out_rate, "combined_score": comb, "n_clusters": n_clust
#         })
#
#         # 4. Sauvegardes locales
#         topic_model.get_topic_info().to_csv(f"{path}/topics_summary.csv", index=False)
#         topic_model.visualize_barcharts(top_n_topics=15).write_html(f"{path}/viz_clusters_words.html")
#
#         # t-SNE de contrôle pour le premier sweep
#         if mcs == MCS_VALUES[0]:
#             tsne = TSNE(n_components=2, random_state=42).fit_transform(reduced_embs)
#             plt.figure(figsize=(10, 7))
#             sns.scatterplot(x=tsne[:,0], y=tsne[:,1], hue=labels, palette='viridis', s=5, alpha=0.6)
#             plt.title(f"Visualisation t-SNE : {sc_name}")
#             plt.savefig(f"{path}/tsne_view.png")
#             plt.close()
#
#     # Étape E : Synthèse du scénario
#     df_sweep = pd.DataFrame(sweep_results)
#     df_sweep.to_csv(f"{OUTPUT_DIR}/{sc_name}/sweep_master_report.csv", index=False)
#
#     # Trouver le meilleur MCS
#     best_mcs = df_sweep.loc[df_sweep['combined_score'].idxmax(), 'mcs']
#     print(f"\n-> Meilleur réglage pour {sc_name} : MCS = {best_mcs}")
#
#     # Étape F : Graphique des performances (Courbes demandées)
#     plt.figure(figsize=(12, 7))
#     plt.plot(df_sweep['mcs'], df_sweep['combined_score'], 'k-o', label='Score Combiné (Cible)', linewidth=3)
#     plt.plot(df_sweep['mcs'], df_sweep['silhouette'], 'r--', label='Silhouette (Séparation)')
#     plt.plot(df_sweep['mcs'], df_sweep['stability'], 'b--', label='Stabilité (Densité)')
#     plt.axvline(x=best_mcs, color='orange', linestyle=':', label=f'Best MCS ({best_mcs})')
#     plt.title(f"Optimisation du Clustering : {sc_name}")
#     plt.xlabel("Min Cluster Size (MCS)"); plt.ylabel("Scores"); plt.legend(); plt.grid(True)
#     plt.savefig(f"{OUTPUT_DIR}/{sc_name}/performance_curves.png")
#     plt.show()
#
# print("\nAnalyse terminée. Tous les résultats sont dans Results/BERTopic/")

In [69]:
# # ============================================================
# # PIPELINE 3 — visualisations finales (heatmap, wordcloud, radar, UMAP)
# # ============================================================
#
# def run_visualization():
#     for sc_name in SCENARIOS:
#         sc_dir  = os.path.join(BASE_DIR, sc_name)
#         out_dir = os.path.join(sc_dir, "cluster_sweep")
#
#         # Chargement embeddings UMAP (utilisés pour le clustering) + meilleurs labels
#         umap_emb_path = os.path.join(sc_dir,  f"{sc_name}_umap_embeddings.npy")  # ✅ embeddings UMAP
#         labels_path   = os.path.join(out_dir, f"{sc_name}_best_labels.npy")
#
#         if not os.path.exists(umap_emb_path) or not os.path.exists(labels_path):
#             log.warning(f"❌ Fichiers manquants pour {sc_name}, skipping")
#             continue
#
#         log.info(f"\n🚀 VISUALISATION : {sc_name}")
#         umap_embeddings = np.load(umap_emb_path)   # ✅ embeddings UMAP réduits
#         best_labels     = np.load(labels_path)
#         log.info(f"📊 Shape embeddings UMAP : {umap_embeddings.shape} | Points : {len(best_labels)}")
#
#         # Reconstruction df_valid avec texte
#         df_sc            = df.copy()
#         df_sc["cluster"] = best_labels
#         valid_cols       = [c for c in SCENARIOS[sc_name] if c in df_sc.columns]
#         df_sc["text"]    = df_sc.apply(lambda row: build_text(row, valid_cols), axis=1)
#         df_valid         = df_sc[df_sc["cluster"] != -1].copy()
#
#         # Colonnes disponibles
#         c_bio    = [c for c in cols_bio_bin   if c in df_valid.columns]
#         c_multi  = [c for c in cols_multi_cat if c in df_valid.columns]
#         c_quanti = [c for c in cols_quanti    if c in df_valid.columns]
#
#         # Wordclouds
#         plot_wordclouds(df_valid, sc_name, out_dir)
#
#         # Radar
#         if c_bio:
#             plot_radar(df_valid, c_bio, sc_name, out_dir)
#
#         # Heatmap
#         if c_bio:
#             plot_heatmap(df_valid, c_bio, sc_name, out_dir)
#
#         # ✅ Double projection UMAP 2D sur embeddings UMAP (euclidean)
#         # Vue globale : grands voisins → structure d'ensemble
#         log.info("🗺️ UMAP 2D - vue globale (n_neighbors=30)...")
#         umap_global = umap.UMAP(
#             n_components=2,
#             n_neighbors=30,
#             min_dist=0.1,
#             metric="euclidean",   # ✅ euclidean car déjà dans espace UMAP réduit
#             random_state=42
#         ).fit_transform(umap_embeddings)
#
#         # Vue locale : petits voisins → séparation fine des clusters
#         log.info("🗺️ UMAP 2D - vue locale (n_neighbors=10)...")
#         umap_local = umap.UMAP(
#             n_components=2,
#             n_neighbors=10,
#             min_dist=0.01,
#             metric="euclidean",   # ✅ idem
#             random_state=42
#         ).fit_transform(umap_embeddings)
#
#         # Plot
#         best_config_path = os.path.join(out_dir, f"{sc_name}_best_config.csv")
#         best_config      = pd.read_csv(best_config_path).iloc[0]
#
#         fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
#         title = (f"BEST MODEL : {sc_name.upper()}\n"
#                  f"Clusters: {int(best_config['n_clusters'])} | mcs: {int(best_config['mcs'])} | "
#                  f"Silhouette: {best_config['silhouette']:.3f} | "
#                  f"Stabilité: {best_config['stability']:.3f} | "
#                  f"Outliers: {best_config['outlier_rate']*100:.1f}%")
#         fig.suptitle(title, fontsize=14, fontweight="bold")
#
#         for ax, coord, name in [
#             (ax1, umap_global, "UMAP Global (n_neighbors=30)"),   # ✅ remplace ancien UMAP
#             (ax2, umap_local,  "UMAP Local  (n_neighbors=10)")    # ✅ remplace t-SNE
#         ]:
#             ax.scatter(coord[best_labels == -1, 0], coord[best_labels == -1, 1],
#                        s=1, color="lightgrey", alpha=0.2, label="Noise")
#             for l in sorted(set(best_labels) - {-1}):
#                 ax.scatter(coord[best_labels == l, 0], coord[best_labels == l, 1],
#                            s=4, alpha=0.6, label=f"C{l}")
#             ax.set_title(name)
#             ax.legend(markerscale=3, fontsize=8, loc="best")
#
#         plt.tight_layout(rect=[0, 0.03, 1, 0.90])
#         plt.savefig(os.path.join(out_dir, f"{sc_name}_FINAL_OPTIMUM.png"), dpi=150)
#         plt.close()
#         log.info(f"💾 Plot sauvegardé : {sc_name}_FINAL_OPTIMUM.png")
#
#         # Signatures CSV
#         res_bio   = df_valid.groupby("cluster")[c_bio].mean() * 100 if c_bio else pd.DataFrame()
#         list_top3 = []
#         for col in c_multi:
#             ct = pd.crosstab(df_valid["cluster"], df_valid[col], normalize="index") * 100
#             def get_top3(row):
#                 top = row.sort_values(ascending=False).head(3)
#                 return " | ".join([f"{n} ({v:.1f}%)" for n, v in top.items() if v > 0])
#             list_top3.append(pd.DataFrame(ct.apply(get_top3, axis=1), columns=[f"Top3_{col}"]))
#
#         quanti_list = []
#         if c_quanti:
#             res_quanti_fmt = pd.DataFrame(index=df_valid.groupby("cluster")[c_quanti].mean().index)
#             for col in c_quanti:
#                 m = df_valid.groupby("cluster")[col].mean().round(1)
#                 s = df_valid.groupby("cluster")[col].std().round(1)
#                 res_quanti_fmt[f"{col}_mean±std"] = m.astype(str) + " ± " + s.astype(str)
#             quanti_list = [res_quanti_fmt]
#
#         final_sigs               = pd.concat([res_bio] + list_top3 + quanti_list, axis=1)
#         counts                   = df_valid["cluster"].value_counts()
#         final_sigs["N_patients"] = counts
#         final_sigs["Poids_%"]    = (counts / len(df) * 100).round(1)
#         if c_bio:
#             final_sigs[c_bio]    = final_sigs[c_bio].round(1)
#         out_csv = os.path.join(out_dir, f"signatures_{sc_name}.csv")
#         final_sigs.sort_values("N_patients", ascending=False).to_csv(out_csv)
#         log.info(f"💾 Signatures sauvegardées : {out_csv}")
#
#     log.info("\n✅ Visualisations terminées !")
#
# if __name__ == "__main__":
#     run_visualization()

In [70]:
# # 1. Chargement
# df = pd.read_csv(CSV_PATH, low_memory=False)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModel.from_pretrained(MODEL_NAME).to(device)
#
# # 2. Boucle sur les Scénarios
# for sc_name, sc_cols in SCENARIOS.items():
#     print(f"\n>>> PROCESSING SCENARIO: {sc_name}")
#
#     # Préparation des données du scénario
#     docs = [build_clinical_text(row, sc_cols) for _, row in df.iterrows()]
#     embeddings = compute_embeddings(docs, tokenizer, model)
#
#     # 3. Sweep sur le min_cluster_size
#     for mcs_val in MCS_VALUES:
#         print(f"--- Running Sweep MCS={mcs_val} ---")
#         topics, t_model = run_bertopic_scenario(sc_name, docs, embeddings, mcs_val)
#
#         # On peut optionnellement sauvegarder les labels dans le DF
#         # df[f'label_{sc_name}_mcs{mcs_val}'] = topics
#
# print("\nAll scenarios and sweeps completed.")

In [71]:
# # ============================================================
# # 3. EXÉCUTION DU SWEEP AVEC GRAPHIQUES
# # ============================================================
#
# for sc_name, sc_cols in SCENARIOS.items():
#     docs = [build_clinical_text(row, sc_cols) for _, row in df.iterrows()]
#     embs = compute_embeddings(docs, tokenizer, model)
#
#     sweep_results = []
#
#     for mcs in MCS_VALUES:
#         print(f"Testing MCS={mcs}...")
#         path = os.path.join(OUTPUT_DIR, sc_name, f"mcs_{mcs}")
#         os.makedirs(path, exist_ok=True)
#
#         # Clustering
#         u_mod = umap.UMAP(n_neighbors=15, n_components=10, metric='cosine', random_state=42)
#         h_mod = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10, gen_min_span_tree=True)
#
#         # On réduit les dimensions pour le clustering
#         reduced_embs = u_mod.fit_transform(embs)
#         labels = h_mod.fit_predict(reduced_embs)
#
#         # Metrics
#         sil, stab, out_rate, comb = calculate_metrics(reduced_embs, labels, h_mod)
#         sweep_results.append({
#             "mcs": mcs, "silhouette": sil, "stability": stab,
#             "outlier_rate": out_rate, "combined_score": comb, "n_clusters": len(np.unique(labels)) - 1
#         })
#
#         # --- VISUEL t-SNE (Seulement pour le MCS par défaut / 1er sweep) ---
#         if mcs == MCS_VALUES[0]:
#             tsne = TSNE(n_components=2, random_state=42).fit_transform(reduced_embs)
#             plt.figure(figsize=(10, 7))
#             sns.scatterplot(x=tsne[:,0], y=tsne[:,1], hue=labels, palette='viridis', s=10)
#             plt.title(f"t-SNE Visualization - {sc_name} (MCS={mcs})")
#             plt.savefig(f"{path}/tsne_clustering.png")
#             plt.close()
#
#     # Sauvegarde du CSV de synthèse
#     df_sweep = pd.DataFrame(sweep_results)
#     df_sweep.to_csv(f"{OUTPUT_DIR}/{sc_name}/sweep_summary.csv", index=False)
#
#     # --- GRAPHIQUE DES COURBES ---
#     plt.figure(figsize=(12, 6))
#     plt.plot(df_sweep['mcs'], df_sweep['combined_score'], 'k-o', label='Combined Score', linewidth=3)
#     plt.plot(df_sweep['mcs'], df_sweep['silhouette'], 'r--', label='Silhouette')
#     plt.plot(df_sweep['mcs'], df_sweep['stability'], 'b--', label='Stability')
#     plt.plot(df_sweep['mcs'], df_sweep['outlier_rate'], 'g--', label='Outlier Rate')
#     plt.xlabel('Min Cluster Size (MCS)')
#     plt.ylabel('Score')
#     plt.title(f'Sweep Metrics Analysis - {sc_name}')
#     plt.legend()
#     plt.grid(True)
#     plt.savefig(f"{OUTPUT_DIR}/{sc_name}/sweep_metrics_plot.png")
#     plt.show()

In [72]:
# # ============================================================
# # CONFIG — imports, chemins, hyperparamètres
# # ============================================================
#
# import logging
# import os
# import warnings
# warnings.filterwarnings("ignore")
#
# # ── THREADING ─────────────────────────────────────────────────────────────────
# os.environ["OPENBLAS_NUM_THREADS"]   = "128"
# os.environ["MKL_NUM_THREADS"]        = "128"
# os.environ["OMP_NUM_THREADS"]        = "128"
# os.environ["NUMEXPR_NUM_THREADS"]    = "128"
# os.environ["VECLIB_MAXIMUM_THREADS"] = "128"
#
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import torch
# import umap
# import hdbscan
# torch.set_num_threads(128)
#
# from tqdm import tqdm
# from sklearn.decomposition import PCA
# from sklearn.manifold import TSNE
# from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
# from transformers import AutoModel, AutoTokenizer
# from wordcloud import WordCloud
#
# # ── LOGS ──────────────────────────────────────────────────────────────────────
# logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
# log = logging.getLogger(__name__)
#
# # ── CHEMINS ───────────────────────────────────────────────────────────────────
# CSV_PATH     = "df_final_binaire_imputed.csv"
# BASE_DIR     = "Results/transformer_clustering"
#
# # ── MODÈLE ────────────────────────────────────────────────────────────────────
# MODEL_NAME   = "yikuan8/Clinical-Longformer"
# BATCH_SIZE   = 64
# MAX_LENGTH   = 512
#
# # ── HYPERPARAMÈTRES ───────────────────────────────────────────────────────────
# UMAP_COMPONENTS_CLUSTERING = 10    # UMAP pour clustering
# UMAP_N_NEIGHBORS           = 15
# UMAP_MIN_DIST_CLUST        = 0.0
# UMAP_N_NEIGHBORS_VIZ       = 30
# UMAP_MIN_DIST_VIZ          = 0.1
# HDBSCAN_MIN_SAMPLES        = 10
# HDBSCAN_MIN_CLUSTER_SIZE   = 500   # valeur par défaut, overridé par le sweep
# SWEEP_MCS_MIN              = 200
# SWEEP_MCS_MAX              = 2000
# SWEEP_MCS_STEP             = 100
# SILHOUETTE_SAMP            = 3000
#
# # ── COLONNES ──────────────────────────────────────────────────────────────────
# cols_veinous_analysis = [
#     'is_hemoglobine', 'is_leucocytes', 'is_formule_leuco', 'is_urea', 'is_creatinine',
#     'is_sodium', 'is_potassium', 'is_platelets', 'is_pt', 'is_aptt',
#     'is_calcium', 'is_ck', 'is_lactates', 'is_troponine', 'is_bnp',
#     'is_ckmb', 'is_ddimer', 'is_crp', 'is_pct', 'is_alat',
#     'is_asat', 'is_bili_total', 'is_lipase', 'is_alp', 'is_iron',
#     'is_ferritin', 'is_calcium_ionized', 'is_aXa_aIIa', 'is_fibrinogen'
# ]
# cols_imaging = [
#     'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
#     'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
#     'radio_interventional_1', 'nuclear_medicine_1'
# ]
# cols_bio_bin = [
#     'is_blood_gas', 'is_aXa_aIIa', 'is_csf', 'is_lactates', 'is_culture',
#     'is_leucocytes', 'is_formule_leuco', 'is_alat', 'is_asat', 'is_bnp',
#     'is_bili_total', 'is_ck', 'is_ckmb', 'is_crp', 'is_calcium_ionized',
#     'is_calcium', 'is_creatinine', 'is_ddimer', 'is_iron', 'is_ferritin',
#     'is_fibrinogen', 'is_hemoglobine', 'is_lipase', 'is_alp', 'is_platelets',
#     'is_potassium', 'is_pct', 'is_sodium', 'is_aptt', 'is_pt',
#     'is_troponine', 'is_urea', 'had_ekg'
# ]
# cols_multi_cat = [
#     'ultrasound_1', 'ultrasound_2', 'ct_scan_1', 'ct_scan_2', 'ct_scan_3',
#     'xray_1', 'xray_2', 'xray_3', 'mri_1', 'mri_2',
#     'radio_interventional_1', 'nuclear_medicine_1'
# ]
# cols_quanti = ['imaging_exam_count', 'bio_exam_count']
#
# # ── SCÉNARIOS ─────────────────────────────────────────────────────────────────
# SCENARIOS = {
#     "advanced_radio_bio_ekg_dispo": [
#         'has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri',
#         'has_radio_interventional', 'has_nuclear_medicine',
#         'has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#         'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#         'imaging_exam_count', 'bio_exam_count',
#     ],
#     "advanced_detailed_radio_bio_ekg_dispo":
#         cols_veinous_analysis + cols_imaging + [
#             'has_culture', 'has_lumbar_puncture', 'has_blood_gas',
#             'had_ekg', 'hospitalization', 'observation_unit', 'inter_facility_transfer',
#             'imaging_exam_count', 'bio_exam_count',
#         ]
# }
#
# # ── VERB MAP ──────────────────────────────────────────────────────────────────
# VERB_MAP = {
#     'has_ultrasound':           "underwent ultrasound",
#     'has_ct_scan':              "underwent CT scan",
#     'has_xray':                 "underwent X-ray",
#     'has_mri':                  "underwent MRI",
#     'has_radio_interventional': "underwent interventional radiology procedure",
#     'has_nuclear_medicine':     "underwent nuclear medicine imaging",
#     'has_blood_test':           "had blood work ordered",
#     'has_culture':              "had microbiological cultures taken",
#     'has_lumbar_puncture':      "underwent lumbar puncture",
#     'has_blood_gas':            "had arterial blood gas analysis",
#     'had_ekg':                  "had an EKG performed",
#     'hospitalization':          "was admitted to the hospital",
#     'observation_unit':         "was placed in observation unit",
#     'inter_facility_transfer':  "was transferred to another facility",
#     'is_hemoglobine':           "hemoglobin was measured",
#     'is_leucocytes':            "white blood cell count was obtained",
#     'is_formule_leuco':         "differential leukocyte count was performed",
#     'is_urea':                  "blood urea was measured",
#     'is_creatinine':            "creatinine was assessed",
#     'is_sodium':                "sodium level was checked",
#     'is_potassium':             "potassium level was checked",
#     'is_platelets':             "platelet count was obtained",
#     'is_pt':                    "prothrombin time was measured",
#     'is_aptt':                  "aPTT was measured",
#     'is_calcium':               "calcium level was assessed",
#     'is_ck':                    "CK was measured",
#     'is_lactates':              "lactate level was obtained",
#     'is_troponine':             "troponin was measured",
#     'is_bnp':                   "BNP was assessed",
#     'is_ckmb':                  "CK-MB was measured",
#     'is_ddimer':                "D-dimer was obtained",
#     'is_crp':                   "CRP was measured",
#     'is_pct':                   "procalcitonin was assessed",
#     'is_alat':                  "ALT was measured",
#     'is_asat':                  "AST was measured",
#     'is_bili_total':            "total bilirubin was assessed",
#     'is_lipase':                "lipase was measured",
#     'is_alp':                   "ALP was measured",
#     'is_iron':                  "serum iron was assessed",
#     'is_ferritin':              "ferritin was measured",
#     'is_calcium_ionized':       "ionized calcium was checked",
#     'is_aXa_aIIa':              "anti-Xa/anti-IIa activity was measured",
#     'is_fibrinogen':            "fibrinogen was assessed",
# }
#
# # ── LOAD DATA ─────────────────────────────────────────────────────────────────
# print("Loading data...")
# df = pd.read_csv(CSV_PATH, low_memory=False)
# print(f"  Shape: {df.shape}")
#
# # ── FEATURE ENGINEERING ───────────────────────────────────────────────────────
# df['observation_unit']      = df['disposition'].str.contains('Obs',      case=False, na=False).astype(int)
# df['inter_facility_transfer'] = df['disposition'].str.contains('transfer', case=False, na=False).astype(int)
#
# print("\nCheck Biology (has_blood_test) :")
# print(df['has_blood_test'].value_counts(normalize=True))
# print("\nRépartition UHCD :")
# print(df['observation_unit'].value_counts())
# print("\nRépartition Transfer :")
# print(df['inter_facility_transfer'].value_counts())

In [73]:
# import pandas as pd
# import numpy as np
# import torch
# from tqdm import tqdm
# from transformers import AutoTokenizer, AutoModel
# from bertopic import BERTopic
# from bertopic.vectorizers import ClassTfidfTransformer
# from sklearn.feature_extraction.text import CountVectorizer
# import hdbscan
# import umap
#
# # ============================================================
# # 1. PRÉPARATION DES TEXTES (La "Forge")
# # ============================================================
#
# def build_text_for_bertopic(row: pd.Series, cols: list) -> str:
#     parts = []
#     for col in cols:
#         val = row[col]
#         # On ignore les vides/nan/0
#         if pd.isna(val) or val == 0 or str(val).upper() in ["NONE", "NAN", ""]:
#             continue
#
#         if isinstance(val, str):
#             # On remplace les espaces par des underscores pour garder le bloc uni
#             clean_val = val.lower().replace(" ", "_")
#             parts.append(clean_val)
#         elif val == 1 or val == 1.0:
#             # Pour les binaires, on prend le nom de la colonne nettoyé
#             clean_col = col.replace('is_','').replace('has_','').replace('had_','')
#             parts.append(clean_col)
#
#     if not parts:
#         # Ton label explicite pour le cluster "vide"
#         return "no_resource_utilization"
#
#     # On construit une phrase simple. Le Longformer comprendra le lien.
#     return "patient " + " ".join(parts)
#
# # Exemple d'application sur un scénario
# # docs = [build_text_for_bertopic(row, SCENARIOS["advanced_detailed_radio_bio_ekg_dispo"]) for _, row in df.iterrows()]
#
# # ============================================================
# # 2. GÉNÉRATION DES EMBEDDINGS (Clinical-Longformer)
# # ============================================================
#
# device = torch.cuda.current_device() if torch.cuda.is_available() else "cpu"
# tokenizer = AutoTokenizer.from_pretrained("yikuan8/Clinical-Longformer")
# model = AutoModel.from_pretrained("yikuan8/Clinical-Longformer").to(device)
#
# def get_embeddings(texts, batch_size=32):
#     all_embs = []
#     model.eval()
#     with torch.no_grad():
#         for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
#             batch = texts[i:i+batch_size]
#             inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
#             outputs = model(**inputs)
#             # Mean Pooling
#             mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
#             sum_embeddings = torch.sum(outputs.last_hidden_state * mask, 1)
#             sum_mask = torch.clamp(mask.sum(1), min=1e-9)
#             all_embs.append((sum_embeddings / sum_mask).cpu().numpy())
#     return np.vstack(all_embs)
#
# # embeddings = get_embeddings(docs)
#
# # ============================================================
# # 3. CONFIGURATION BERTopic
# # ============================================================
#
# # A. UMAP pour la réduction de dimension (avant clustering)
# umap_model = umap.UMAP(
#     n_neighbors=15,
#     n_components=10,
#     min_dist=0.0,
#     metric='cosine',
#     random_state=42
# )
#
# # B. HDBSCAN pour le clustering (ton moteur habituel)
# hdbscan_model = hdbscan.HDBSCAN(
#     min_cluster_size=500,
#     min_samples=10,
#     metric='euclidean',
#     cluster_selection_method='eom'
# )
#
# # C. Vectorizer qui respecte tes underscores (_)
# # Le pattern (?u)\b\w[\w_]+\b permet de ne pas couper "no_resource_utilization"
# vectorizer_model = CountVectorizer(
#     token_pattern=r"(?u)\b\w[\w_]+\b",
#     stop_words="english"
# )
#
# # D. Transformer c-TF-IDF (Nettoyage automatique des mots "Patient", etc.)
# ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
#
# # E. Assemblage du modèle BERTopic
# topic_model = BERTopic(
#     embedding_model=None,        # On passe nos propres embeddings calculés plus haut
#     umap_model=umap_model,
#     hdbscan_model=hdbscan_model,
#     vectorizer_model=vectorizer_model,
#     ctfidf_model=ctfidf_model,
#     verbose=True
# )
#
# # ============================================================
# # 4. EXÉCUTION ET VISUALISATION
# # ============================================================
#
# # On lance le fitting
# topics, probs = topic_model.fit_transform(docs, embeddings)
#
# # --- RÉSULTATS ---
#
# # 1. Tableau récapitulatif des clusters
# print(topic_model.get_topic_info())
#
# # 2. Visualisation des mots-clés par cluster (Le top 8 des mots par groupe)
# fig_bar = topic_model.visualize_barcharts(top_n_topics=15, n_words=8)
# fig_bar.write_html("results_topics_bars.html") # Sauvegarde en interactif
#
# # 3. Visualisation de la proximité des clusters
# fig_intertopic = topic_model.visualize_topics()
# fig_intertopic.write_html("results_intertopic_map.html")
#
# # 4. Export des résultats dans ton DataFrame original
# df['cluster_bertopic'] = topics